# ECE1512 Project A — ONE notebook, C16 → C17 → BRACS

This notebook keeps the course's **original ABMIL, DimReduction, AttentionGated, Classifier1FC, train_one_epoch and evaluate** definitions. It reuses the extra ablation model from your own uploaded notebook. The only new part is orchestration: datasets and stages execute sequentially in this **single kernel**, with visible per-epoch progress and saved stage folders. **No pip install and no repeated raw-WSI visualizations.**

Original C16/C17/BRACS notebooks are **not modified**. Choose the experiment plan in the next cell. For unattended execution use the supplied `run_one_notebook.py` runner, which writes an `...EXECUTED.ipynb` copy after every completed cell.

**Research integrity:** select new architectures on validation only; leave `EVALUATE_TEST=False` until after model selection.

In [1]:
# CELL 1 — USER CONTROLS. Change only these settings.
from pathlib import Path
PROJECT_DIR = Path('.').resolve()       # original course project root with config/, dataloader.py
DATASETS = ('C16', 'C17', 'BRACS')       # runs sequentially in this order
EXPERIMENT_PLAN = 'focused'            # 'focused', 'all_singles_pairs', 'all', or 'custom'
RUN_EXISTING_STAGES = False            # True: run original A0–A4 again; False: preserve prior results
RUN_EXTRA_ABLATIONS = True
EPOCH_OVERRIDE = None                  # 2 for quick debugging; None = original 30 epochs
SKIP_COMPLETED_ABLATIONS = True
RESUME_INCOMPLETE_STAGES = True        # resume from latest saved epoch when possible
EVALUATE_TEST = False                  # new ablations: validation first; no automatic test peeking
TOP_K_A4 = 16
A1_TAU, A2_GAMMA = 0.5, 1.0
A3_LAMBDA, A3_TEMPERATURE = 0.05, 0.2
# If you only want several experiments, set EXPERIMENT_PLAN='custom',
# and edit ABLATION_CUSTOM in the experiment definition cell below.
print('Project root:', PROJECT_DIR)
print('Datasets:', DATASETS, '| Plan:', EXPERIMENT_PLAN)

Project root: C:\Users\anest\Desktop\UofT\Digital Image Processing\Project A\Project_A_Starter_Code\Project_A_Starter_Code
Datasets: ('C16', 'C17', 'BRACS') | Plan: focused


In [2]:
# CELL 2 — imports and ORIGINAL course loader utilities (no pip install cell).
import os, sys, gc, json, time, random, datetime, warnings, traceback
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')  # save figures; do not block overnight training
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score, f1_score)
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
from dataloader import build_HDF5_feat_dataset, mil_collate_fn
from utils import load_config, seed_everything, infer_feature_dim, classification_metrics
print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():print('CUDA device:', torch.cuda.get_device_name(0))
else:print('WARNING: CUDA unavailable — training will likely be much slower than your previous GPU runs.')
CONFIG_FILES = {
  'C16':'camelyon16_medical_ssl_config.yml',
  'C17':'camelyon17_medical_ssl_config.yml',
  'BRACS':'bracs_medical_ssl_config.yml',
}
RESULTS_ROOT = PROJECT_DIR
PROGRESS_FILE = PROJECT_DIR / 'all_datasets_progress.log'
def progress(message):
    line = f'[{datetime.datetime.now().isoformat(timespec="seconds")}] {message}'
    print(line, flush=True)
    with PROGRESS_FILE.open('a',encoding='utf-8') as f: f.write(line+'\n')
progress('Unified run started. Device: '+str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')))

c:\Users\anest\anaconda3\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.6.0+cu126 | CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU
[2026-11-03T04:41:55] Unified run started. Device: cuda


## Original course ABMIL (unchanged)
The next cell is the original `Classifier1FC`, `DimReduction`, `AttentionGated`, and `ABMIL` from your C16 notebook, without modifications.

In [3]:
class Classifier1FC(nn.Module):
    def __init__(self, n_channels, n_classes, droprate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(droprate) if droprate > 0 else nn.Identity()
        # TODO 2A: one linear layer from n_channels to n_classes.
        self.fc = nn.Linear(in_features = n_channels, out_features = n_classes) # TODO('Create a linear classifier')

    def forward(self, x):
        # TODO 2B: apply dropout, then the classifier.
        return self.fc(self.dropout(x)) # TODO('Return classifier output')


class DimReduction(nn.Module):
    def __init__(self, n_channels, m_dim):
        super().__init__()
        # TODO 2C: project patch features to m_dim and use ReLU.
        self.fc = nn.Linear(in_features = n_channels, out_features = m_dim) # TODO('Create Linear layer')
        self.relu = nn.ReLU() # TODO('Create ReLU')

    def forward(self, x):
        # TODO 2D: linear projection followed by ReLU.
        return self.relu(self.fc(x)) # TODO('Return ReLU(linear(x))')


class AttentionGated(nn.Module):
    def __init__(self, L=256, D=128, K=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_weights_w = nn.Linear(D, K)

    def forward(self, x):
        # x: [B, N, L]
        # TODO 2E: implement the two gated branches.
        a_v = self.attention_V(x) # TODO('Compute tanh branch')
        a_u = self.attention_U(x) # TODO('Compute sigmoid branch')

        a = self.attention_weights_w(a_v * a_u)
        return a.transpose(1, 2)             


class ABMIL(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.dimreduction = DimReduction(conf.D_feat, conf.D_inner)
        self.attention = AttentionGated(conf.D_inner, conf.D_attn, K=1)
        self.classifier = Classifier1FC(conf.D_inner, conf.n_class, conf.droprate)

    def forward(self, x, return_attention=False):
        # x: [B, N, D_feat]
        med_feat = self.dimreduction(x)       # [B, N, D_inner]
        a_logits = self.attention(med_feat)   # [B, 1, N]


        # TODO 2G: normalize attention logits over the instance dimension.
        attention = F.softmax(input = a_logits, dim = -1) # TODO('Apply softmax over patches')

        # TODO 2H: weighted aggregation. torch.bmm is useful here.
        bag_feat = torch.bmm(input = attention, mat2 = med_feat) # [B, 1, D_inner] # TODO('Aggregate patch features into one slide representation')
        bag_feat = bag_feat.squeeze(1) # # [B, D_inner]

        # TODO 2I: classify the slide representation.
        logits = self.classifier(bag_feat) # [B, n_class] # TODO('Compute slide-level logits')

        if return_attention:
            return logits, attention
        return logits

In [4]:
def train_one_epoch(model, criterion, data_loader, optimizer, device):
    # TODO 4A: put the model in training mode.
    model.train() # TODO('Call model.train()')

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for data in data_loader:
        # TODO 4B: move input and labels to the selected device.
        bag    = data["input"].to(device=device, dtype=torch.float32) # TODO('Move data["input"] to device as float32')
        labels = data["label"].to(device=device, dtype=torch.long)    # TODO('Move data["label"] to device as long')

        # TODO 4C: clear gradients from the previous iteration.
        optimizer.zero_grad() # TODO('optimizer zero grad')

        # TODO 4D: forward pass.
        logits = model(bag) # TODO('Pass the feature bag to the ABMIL')

        # TODO 4E: compute the loss.
        loss = criterion(logits, labels) # TODO('criterion(logits, labels)')

        # TODO 4F: back-propagate gradients.
        loss.backward() # TODO('loss.backward()')

        # TODO 4G: update the model parameters.
        optimizer.step() # TODO('optimizer.step()')

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return {
        'loss': total_loss / total_examples,
        'acc': 100.0 * total_correct / total_examples,
    }


@torch.no_grad()
def evaluate(model, criterion, data_loader, device, conf):
    # TODO 5A: switch to evaluation mode.
    model.eval() # TODO('Call model.eval()')

    total_loss = 0.0
    total_examples = 0
    probabilities_all = []
    labels_all = []

    for data in data_loader:
        # TODO 5B: move data to device.
        bag = data["input"].to(device=device, dtype=torch.float32) # TODO('Move input to device')
        labels = data["label"].to(device=device, dtype=torch.long) # TODO('Move labels to device')

        # TODO 5C: forward pass and loss.
        logits = model(bag) # TODO('Compute logits')
        loss = criterion(logits, labels) # TODO('Compute loss')

        # TODO 5D: convert class logits into probabilities.
        probabilities = F.softmax(logits, dim=1) # TODO('Apply softmax over classes')

        total_loss += loss.item() * labels.shape[0]
        total_examples += labels.shape[0]
        probabilities_all.append(probabilities)
        labels_all.append(labels)

    probabilities_all = torch.cat(probabilities_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)
    metrics = classification_metrics(probabilities_all, labels_all, conf.n_class)
    metrics['loss'] = total_loss / total_examples
    return metrics

## Reused reporting utilities
Saving and scoring use the same conventions as your original notebooks; the current dataset and its train-label counts are assigned before each dataset starts.

In [5]:
def to_python(x):
    if isinstance(x, dict): return {str(k): to_python(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [to_python(v) for v in x]
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x) if np.isfinite(x) else None
    if isinstance(x, torch.Tensor): return to_python(x.detach().cpu().numpy().tolist())
    if isinstance(x, (float,)): return x if np.isfinite(x) else None
    return x

def stage_dir(stage):
    d = RESULTS_ROOT / f'results_{DATASET}_{stage}'  # EXACT requested naming convention
    d.mkdir(parents=True, exist_ok=True)
    return d

def train_label_counts():
    # ADDED A1: ONLY training slide labels, never validation/test labels.
    # Fast metadata-only path for the original course HDF5 format.
    counts = np.zeros(int(conf.n_class), dtype=np.int64)
    try:
        import h5py
        from dataloader import _map_label
        with h5py.File(conf.data_dir, 'r') as handle:
            for sid in train_data.data_names:
                raw = int(handle[sid].attrs['label'])
                y = int(_map_label(raw, conf.dataset, int(conf.n_class)))
                counts[y] += 1
    except (ImportError, AttributeError, KeyError, OSError, TypeError) as exc:
        # Fallback works with other course-style Dataset implementations.
        print('Using train_data label fallback:', type(exc).__name__)
        counts[:] = 0
        for j in range(len(train_data)):
            label = train_data[j]['label']
            counts[int(torch.as_tensor(label).item())] += 1
    if not np.all(counts > 0):
        raise ValueError(f'Expected at least one training slide per class, got {counts}')
    return counts


def collect_stage_predictions(model, loader, include_attention=False):
    # ADDED: same forward logits as A0; NEVER applies adjusted logits at test time.
    model.eval()
    ys, probs, ids = [], [], []
    first_attention = None
    first_slide = None
    with torch.no_grad():
        for batch in loader:
            x = batch['input'].to(device, dtype=torch.float32)
            y = batch['label'].to(device, dtype=torch.long).reshape(-1)
            if include_attention and first_attention is None:
                logits, attention = model(x, return_attention=True)
                first_attention = attention[0].detach().cpu().numpy()
                sid = batch.get('slide_id', ['unknown'])
                first_slide = str(sid[0] if isinstance(sid, (tuple, list)) else sid)
            else:
                logits = model(x)
            ys.append(y.cpu().numpy())
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            sid = batch.get('slide_id', None)
            if isinstance(sid, (tuple, list)):
                ids.extend([str(v) for v in sid])
            elif sid is not None:
                ids.extend([str(sid)] * y.shape[0])
            else:
                ids.extend([f'row_{len(ids)+v}' for v in range(y.shape[0])])
    yy = np.concatenate(ys).astype(int)
    pp = np.concatenate(probs)
    return yy, pp, ids, first_attention, first_slide

def prediction_metrics(y, p):
    predicted = p.argmax(axis=1)
    per_auc = {}
    for c in CLASS_IDS:
        binary = (y == c).astype(int)
        per_auc[f'class_{c}'] = (float(roc_auc_score(binary, p[:, c]))
                                 if len(np.unique(binary)) == 2 else None)
    valid_auc = [z for z in per_auc.values() if z is not None]
    return {
        'accuracy': float(np.mean(y == predicted) * 100.0),
        'macro_f1': float(f1_score(y, predicted, labels=CLASS_IDS, average='macro', zero_division=0)),
        'macro_ovr_auc': float(np.mean(valid_auc)) if valid_auc else None,
        'per_class_auc': per_auc,
        'class_recall': {str(c): float(np.mean(predicted[y == c] == c)) if np.any(y == c) else None
                         for c in CLASS_IDS},
        'n_slides': int(len(y))
    }

def save_predictions(y, p, slide_ids, directory, split):
    pd.DataFrame({
        'slide_id': slide_ids, 'y_true': y, 'y_pred': p.argmax(axis=1),
        **{f'prob_{c}': p[:, c] for c in CLASS_IDS}
    }).to_csv(directory / f'{split}_predictions.csv', index=False)

def plot_stage_figures(stage, directory, hist, y, p, att, slide_id, best_ep):
    epochs = hist['epoch'].to_numpy() + 1
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(epochs, hist['train_loss'], label='Training objective')
    ax.plot(epochs, hist['val_loss'], label='Validation CE (raw logits)')
    ax.axvline(best_ep + 1, linestyle='--', color='gray', label=f'Best epoch {best_ep+1}')
    ax.set(xlabel='Epoch', ylabel='Loss', title=f'{DATASET} {stage}: learning curves')
    ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
    fig.savefig(directory / 'learning_curves.png', dpi=180)
    plt.close(fig)

    yp = p.argmax(axis=1)
    cm = confusion_matrix(y, yp, labels=CLASS_IDS)
    pd.DataFrame(cm, index=CLASS_IDS, columns=CLASS_IDS).to_csv(directory / 'confusion_matrix.csv')
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_IDS).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{DATASET} {stage}: held-out test confusion matrix'); fig.tight_layout()
    fig.savefig(directory / 'confusion_matrix.png', dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6.2, 4.7))
    n_curves = 0
    for c in CLASS_IDS:
        binary = (y == c).astype(int)
        if len(np.unique(binary)) < 2: continue  # AUROC undefined when class absent.
        fpr, tpr, _ = roc_curve(binary, p[:, c])
        auc_val = roc_auc_score(binary, p[:, c]); n_curves += 1
        ax.plot(fpr, tpr, label=f'{c} (AUC={auc_val:.3f})')
    ax.plot([0,1], [0,1], '--', color='gray', linewidth=1)
    ax.set(xlabel='False-positive rate', ylabel='True-positive rate',
           title=f'{DATASET} {stage}: test one-vs-rest ROC')
    if n_curves: ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(directory / 'roc_curves.png', dpi=180)
    plt.close(fig)

    recalls = [float(np.mean(yp[y == c] == c)) if np.any(y == c) else np.nan for c in CLASS_IDS]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([str(c) for c in CLASS_IDS], [0 if np.isnan(z) else z * 100 for z in recalls])
    ax.set(ylim=(0, 100), xlabel='Class', ylabel='Recall (%)',
           title=f'{DATASET} {stage}: held-out recall per class')
    fig.tight_layout(); fig.savefig(directory / 'per_class_recall.png', dpi=180)
    plt.close(fig)

    if att is not None:
        # This is a ranked patch-attention plot, NOT a spatial WSI heatmap.
        # For class-conditioned models, plot the predicted class's attention distribution.
        plot_att = np.asarray(att)
        # Determine the first test slide's predicted class using its saved prediction.
        idx = int(yp[0]) if plot_att.shape[0] > 1 else 0
        weights = plot_att[idx]
        ranked = np.sort(weights)[::-1]
        fig, ax = plt.subplots(figsize=(7, 3.8))
        ax.plot(np.arange(1, min(150, len(ranked)) + 1), ranked[:150])
        ax.set(xlabel='Ranked patch index', ylabel='Normalized attention weight',
               title=f'{DATASET} {stage}: top-attended patches ({slide_id})')
        fig.tight_layout(); fig.savefig(directory / 'attention_ranked.png', dpi=180)
        plt.close(fig)
        top = np.argsort(weights)[::-1][:min(50, len(weights))]
        pd.DataFrame({'slide_id': slide_id, 'predicted_class': int(yp[0]),
                      'patch_index': top, 'attention_weight': weights[top]}).to_csv(
                          directory / 'attention_top_patches.csv', index=False)

In [6]:
# Anestis

# ADDED A1: only a training LOSS. Reuses ORIGINAL ABMIL(conf) and ALL its components.
class LogitAdjustedCrossEntropy(nn.Module):
    def __init__(self, counts, tau=0.5):
        super().__init__()
        prior = torch.tensor(counts, dtype=torch.float32)
        prior = prior / prior.sum()
        self.register_buffer('log_prior', prior.log())
        self.tau = float(tau)

    def forward(self, logits, labels):
        # ADDED A1: train-time logit adjustment; do not modify model.forward().
        return F.cross_entropy(logits + self.tau * self.log_prior[None, :], labels)

## Ablation specifications and original modular architecture
Every letter is independently switchable: L=logit adjustment, H=class heads, P=prototype-guided attention, R=prototype loss, S=attention retention, C=selected-token context. Existing A0–A4 are mapped to these same switches and run only when `RUN_EXISTING_STAGES=True`.

In [7]:
# EXTRA ABLATIONS — configuration (separate from your original A0–A4)
# Each letter represents a switch, not an incremental fine-tuning checkpoint.
# L: logit adjustment (A1 alone)
# H: class-specific GATED heads WITHOUT prototypes
# P: prototype-guided attention WITHOUT class-specific gated heads
# R: prototype discrimination auxiliary loss WITHOUT prototype-guided attention
# S: strong-attention evidence retention for classification
# C: selected-token contextual MHA WITHOUT the other extensions
# A pair combines exactly two of those independent switches.
# P-only expands the ORIGINAL shared gated scores using a class-wise cosine prototype score.
# R-only compares the original one-vector ABMIL slide representation with class prototypes.
# S-only retains high-attention evidence; no prototype supervision is required.
# C-only adds selected-token context to the original single-head ABMIL slide vector.

from itertools import combinations
from dataclasses import dataclass, asdict, replace
from typing import Tuple

# 'focused' = the most interpretable follow-ups, 'all_singles_pairs' = 6 + 15 runs,
# 'custom' = use ABLATION_CUSTOM list below. You can change this in each notebook.
ABLATION_PLAN = EXPERIMENT_PLAN
# RUN_EXTRA_ABLATIONS configured in CELL 1
# EPOCH_OVERRIDE configured in CELL 1
# SKIP_COMPLETED_ABLATIONS configured in CELL 1
TEST_NEW_ABLATIONS = EVALUATE_TEST
FINAL_ABLATION_NAME = None

@dataclass(frozen=True)
class AblationSpec:
    name: str
    enabled: Tuple[str,...] = ()
    selection: str = 'none'      # none, top_fraction, attention_mass, top_k
    fraction: float = 0.20      # 0.20 = highest 20% (NOT the 80th percentile value)
    mass: float = 0.80          # smallest set with >=80% of attention mass
    top_k: int = 16
    min_keep: int = 8
    blend: float = 0.50         # 0 full pooling / 1 hard selected pooling
    selection_scope: str = 'pool'  # pool or prototype_only (requires R)
    selection_rank: str = 'combined' # combined or gated
    context_order: str = 'after_pool' # after_pool or before_pool
    context_scale: float = 0.20
    proto_gamma: float = 1.0
    proto_lambda: float = 0.05
    tau: float = 0.50
    temperature: float = 0.20
    seed_offset: int = 0

    def __post_init__(self):
        valid = set('LHPRSC')
        if len(set(self.enabled)) != len(self.enabled) or not set(self.enabled) <= valid:
            raise ValueError(f'Unknown/repeated mechanisms: {self.enabled}')
        if self.selection_scope not in ('pool','prototype_only'):
            raise ValueError('selection_scope must be pool/prototype_only')
        if 'S' in self.enabled and self.selection == 'none':
            object.__setattr__(self, 'selection', 'attention_mass')
        if self.selection_scope == 'prototype_only' and not {'S','R'} <= set(self.enabled):
            raise ValueError('prototype_only requires S+R')
        if not (0.0 < self.fraction <= 1.0 and 0.0 < self.mass <= 1.0):
            raise ValueError('fraction and mass must be in (0,1]')
        if not (0.0 <= self.blend <= 1.0):
            raise ValueError('blend must be in [0,1]')

ALL_KEYS = ('L','H','P','R','S','C')
ALL_SINGLE_PAIRS = [
    AblationSpec('X_'+''.join(keys), enabled=keys)
    for count in (1,2)
    for keys in combinations(ALL_KEYS, count)
]

# Main study: original A0–A4 are already available in the starter notebook.
# The focused plan dissects the bundled changes and compares the retention options.
FOCUSED_ABLATIONS = [
    AblationSpec('X_H', ('H',)),
    AblationSpec('X_P', ('P',)),
    AblationSpec('X_R', ('R',)),
    AblationSpec('X_S', ('S',)),
    AblationSpec('X_C', ('C',)),
    AblationSpec('X_HP', ('H','P')),
    AblationSpec('X_HR', ('H','R')),
    AblationSpec('X_PR', ('P','R')),
    AblationSpec('X_LHP', ('L','H','P')),
    AblationSpec('X_LHPR', ('L','H','P','R')),
    AblationSpec('X_LHPRS_mass80', ('L','H','P','R','S'), selection='attention_mass', mass=.80, selection_scope='prototype_only'),
    AblationSpec('X_LHPRS_top20', ('L','H','P','R','S'), selection='top_fraction', fraction=.20, selection_scope='prototype_only'),
    AblationSpec('X_LHPRS_top80', ('L','H','P','R','S'), selection='top_fraction', fraction=.80, selection_scope='prototype_only'),
    AblationSpec('X_LHPRS_top16', ('L','H','P','R','S'), selection='top_k', top_k=16, selection_scope='prototype_only'),
    # ORDER ablation: select using gated evidence BEFORE prototype similarity vs combined evidence.
    AblationSpec('X_LHPRS_gatedmass80', ('L','H','P','R','S'), selection='attention_mass', mass=.80, selection_scope='prototype_only', selection_rank='gated'),
    # Alternative: apply evidence selection to slide CLASSIFICATION itself (50% blend), not only R loss.
    AblationSpec('X_LHPRS_poolmass80', ('L','H','P','R','S'), selection='attention_mass', mass=.80, selection_scope='pool', blend=.50),
    AblationSpec('X_LHPRC_low', ('L','H','P','R','C'), context_scale=.05),
    AblationSpec('X_LHPRC_before', ('L','H','P','R','C'), context_order='before_pool', context_scale=.20),
]

# Add optional full factorial pairs on top of specialized threshold/order tests.
ABLATION_CUSTOM = [
    AblationSpec('X_LHPRS_mass90', ('L','H','P','R','S'), selection='attention_mass', mass=.90, selection_scope='prototype_only'),
    AblationSpec('X_LHPR_lam001', ('L','H','P','R'), proto_lambda=.01),
    AblationSpec('X_LHPR_lam010', ('L','H','P','R'), proto_lambda=.10),
    AblationSpec('X_LHPR_seed43', ('L','H','P','R'), seed_offset=1),
    AblationSpec('X_LHPR_seed44', ('L','H','P','R'), seed_offset=2),
]

if ABLATION_PLAN == 'focused':
    ABLATION_SPECS = FOCUSED_ABLATIONS
elif ABLATION_PLAN == 'all_singles_pairs':
    ABLATION_SPECS = ALL_SINGLE_PAIRS
elif ABLATION_PLAN == 'custom':
    ABLATION_SPECS = ABLATION_CUSTOM
elif ABLATION_PLAN == 'all':
    ABLATION_SPECS = list({s.name:s for s in (ALL_SINGLE_PAIRS+FOCUSED_ABLATIONS+ABLATION_CUSTOM)}.values())
else:
    raise ValueError('ABLATION_PLAN must be focused/all_singles_pairs/all/custom')

# Stages are independent: do not carry weights from one experiment to another.
print('Ablation plan:', ABLATION_PLAN, '| new runs:', len(ABLATION_SPECS))
print('Enabled:', [s.name for s in ABLATION_SPECS] if RUN_EXTRA_ABLATIONS else [])


# ORIGINAL A0–A4 modeled using the SAME original modules, with their respective additions.
ORIGINAL_STAGE_SPECS = [
    AblationSpec('A0'),
    AblationSpec('A1', ('L',)),
    AblationSpec('A2', ('L','H','P')),
    AblationSpec('A3', ('L','H','P','R')),
    AblationSpec('A4', ('L','H','P','R','C')),
]
STAGE_SPECS = (ORIGINAL_STAGE_SPECS if RUN_EXISTING_STAGES else []) + (ABLATION_SPECS if RUN_EXTRA_ABLATIONS else [])
print('Training runs per dataset:',len(STAGE_SPECS),'| Three datasets:',len(STAGE_SPECS)*len(DATASETS))

Ablation plan: focused | new runs: 18
Enabled: ['X_H', 'X_P', 'X_R', 'X_S', 'X_C', 'X_HP', 'X_HR', 'X_PR', 'X_LHP', 'X_LHPR', 'X_LHPRS_mass80', 'X_LHPRS_top20', 'X_LHPRS_top80', 'X_LHPRS_top16', 'X_LHPRS_gatedmass80', 'X_LHPRS_poolmass80', 'X_LHPRC_low', 'X_LHPRC_before']
Training runs per dataset: 18 | Three datasets: 54


In [8]:
# EXTRA MODEL — modular ablation reuses YOUR original ABMIL modules.
# REUSED: ABMIL, DimReduction, AttentionGated, Classifier1FC and projection weights.
# NEW: only independent switches for attention heads, prototypes, retention and context.

class AblationABMIL(ABMIL):
    def __init__(self, conf, spec):
        super().__init__(conf)  # original A0 creates DimReduction + AttentionGated(K=1) + Classifier1FC
        self.spec = spec
        self.uses = set(spec.enabled)
        self.C = int(conf.n_class)
        self.D = int(conf.D_inner)
        self.classwise = ('H' in self.uses) or ('P' in self.uses)
        if 'H' in self.uses:
            # ADDED H: same original AttentionGated class, now K=C.
            self.attention = AttentionGated(conf.D_inner, conf.D_attn, K=self.C)
        if 'P' in self.uses or 'R' in self.uses:
            # ADDED P/R: latent vectors learned from slide labels (NOT patch labels).
            self.prototypes = nn.Parameter(torch.randn(self.C, self.D) * 0.05)
        if 'C' in self.uses:
            # ADDED C: context on selected tokens only.
            self.context_norm = nn.LayerNorm(self.D)
            self.context_attention = nn.MultiheadAttention(self.D, num_heads=4, batch_first=True)

    def score_patches(self, h):
        gated = self.attention(h)                       # [B,1,N] or [B,C,N]
        if 'P' in self.uses:
            cosine = (F.normalize(h, dim=-1) @
                      F.normalize(self.prototypes, dim=-1).T).transpose(1,2)
            # P-only: broadcast one ORIGINAL gated head to C prototype-conditioned heads.
            scores = gated + self.spec.proto_gamma * cosine
        else:
            scores = gated
        return gated, scores

    def retain_mask(self, probabilities):
        # Returns [B,H,N] bool mask; selection based on detached rankings only.
        B,H,N = probabilities.shape
        sp = self.spec
        if 'S' not in self.uses:
            return torch.ones_like(probabilities, dtype=torch.bool)
        ranked_weights, order = probabilities.detach().sort(dim=-1, descending=True)
        if sp.selection == 'top_k':
            k = min(N, max(sp.min_keep, sp.top_k))
            by_rank = (torch.arange(N, device=probabilities.device) < k)[None,None,:]
            sorted_mask = by_rank.expand(B,H,-1)
        elif sp.selection == 'top_fraction':
            k = min(N, max(sp.min_keep, int(np.ceil(sp.fraction * N))))
            by_rank = (torch.arange(N, device=probabilities.device) < k)[None,None,:]
            sorted_mask = by_rank.expand(B,H,-1)
        elif sp.selection == 'attention_mass':
            # Keep first patch *crossing* cumulative mass, not only the ones before it.
            prior_mass = ranked_weights.cumsum(-1) - ranked_weights
            sorted_mask = prior_mass < sp.mass
            sorted_mask[..., :min(N, sp.min_keep)] = True
        else:
            raise ValueError(f'Unexpected selection mode {sp.selection}')
        mask = torch.zeros_like(probabilities, dtype=torch.bool)
        return mask.scatter(-1, order, sorted_mask)

    def forward(self, x, return_attention=False, return_aux=False):
        h = self.dimreduction(x)                     # REUSE original A0 projection [B,N,D]
        gated, scores = self.score_patches(h)
        B,N,D = h.shape
        H = scores.shape[1]
        context = None
        chosen = None
        if 'C' in self.uses:
            # Select top-k from every *currently available* attention stream.
            # For C-only, there is one original A0 stream, not C class streams.
            k = min(16, N)
            chosen = scores.detach().topk(k, dim=-1).indices.reshape(B,-1)
            chosen_h = h.gather(1, chosen.unsqueeze(-1).expand(-1,-1,D))
            tokens = self.context_norm(chosen_h)
            context, _ = self.context_attention(tokens,tokens,tokens, need_weights=False)
            if self.spec.context_order == 'before_pool':
                # ORDER ABLATION: insert contextual deltas into patch features BEFORE
                # recomputing class evidence and pooling. Repeated indices average.
                delta = torch.zeros_like(h).scatter_add(
                    1, chosen.unsqueeze(-1).expand(-1,-1,D), context)
                counts = torch.zeros(B,N,1,device=h.device,dtype=h.dtype).scatter_add(
                    1, chosen.unsqueeze(-1), torch.ones(B,chosen.shape[1],1,device=h.device,dtype=h.dtype))
                h = h + self.spec.context_scale * delta / counts.clamp_min(1)
                gated, scores = self.score_patches(h)
                H = scores.shape[1]
            elif self.spec.context_order != 'after_pool':
                raise ValueError('context_order must be before_pool / after_pool')

        attention = scores.softmax(dim=-1)            # same normalizer as A0
        z_full = torch.bmm(attention, h)              # same A0 pooling; [B,H,D]
        retained = None
        selected_z = z_full
        if 'S' in self.uses:
            if self.spec.selection_rank == 'gated':
                rank_scores = gated.expand(-1,H,-1).softmax(-1)
            else:
                rank_scores = attention
            retained = self.retain_mask(rank_scores)
            selected_attention = attention * retained.to(attention.dtype)
            selected_attention = selected_attention / selected_attention.sum(-1, keepdim=True).clamp_min(1e-10)
            selected_z = torch.bmm(selected_attention, h)
            # For prototype_only: leave classifier pooling unchanged; use selected_z in R loss.
            if self.spec.selection_scope == 'prototype_only':
                z = z_full
            else:
                z = ((1-self.spec.blend)*z_full + self.spec.blend*selected_z)
        else:
            z = z_full

        if ('C' in self.uses) and self.spec.context_order == 'after_pool':
            # ORIGINAL A4 order: add contextual residual to pooled class features.
            gathered = scores.gather(2, chosen[:,None,:].expand(-1,H,-1))
            context_weights = gathered.softmax(-1)
            z = z + self.spec.context_scale * torch.bmm(context_weights, context)

        if self.classwise:
            # REUSE original Classifier1FC, then keep class c from its class-c vector.
            logits = self.classifier(z).diagonal(dim1=1,dim2=2)
        else:
            logits = self.classifier(z.squeeze(1)) # same original A0 classifier

        # The R loss compares the correct slide's pooled evidence with prototypes.
        # R-only uses the single A0 representation, shared across class comparisons.
        proto_source = selected_z if ('S' in self.uses and self.spec.selection_scope == 'prototype_only') else z
        auxiliary = {'class_bag_features': z,
                     'prototype_features': proto_source,
                     'selection_count': retained.sum(-1) if retained is not None else torch.full(
                         (B,H),N,device=h.device,dtype=torch.long),
                     'n_patches': N}
        if return_attention and return_aux: return logits, attention, auxiliary
        if return_attention: return logits, attention
        if return_aux: return logits, auxiliary
        return logits

# Model output retains the starter API: model(x) -> raw [B,C] logits;
# model(x, return_attention=True) -> logits, attention[B,H,N].

In [9]:
# EXTRA TRAINING — REUSE your starter evaluate(), loaders, AdamW settings and
# your original results_{DATASET}_{Stage} naming convention.
# Unlike the older exporter, NEW experiment selection uses VALIDATION ONLY by default.
# Existing A0–A4 results are not overwritten by names beginning X_.

import traceback, datetime, csv
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score

# ABLATION_EPOCHS is set for each dataset.

def aux_prototype_loss(net, aux, y, temperature):
    Z = aux['prototype_features']
    if Z.shape[1] == int(conf.n_class):
        pooled = Z[torch.arange(len(y),device=y.device), y]
    else:
        pooled = Z[:,0]   # original A0 pooled representation (R-only)
    proto_logits = (F.normalize(pooled,dim=-1) @
                    F.normalize(net.prototypes,dim=-1).T) / temperature
    return F.cross_entropy(proto_logits, y)

@torch.no_grad()
def collect_ablation_details(net, loader, directory, split='validation'):
    # Save predictions, per-class attention mass and number of selected patches.
    net.eval()
    all_y, all_p, slides, counts = [], [], [], []
    for batch in loader:
        x = batch['input'].to(device,dtype=torch.float32)
        y = batch['label'].to(device,dtype=torch.long).reshape(-1)
        logits, att, aux = net(x, return_attention=True, return_aux=True)
        probs = logits.softmax(-1)
        B = x.shape[0]
        names = batch.get('slide_id', None)
        if isinstance(names,(list,tuple)): names=[str(s) for s in names]
        elif names is None: names=[f'row_{len(all_y)+b}' for b in range(B)]
        else: names=[str(names)] * B
        for b in range(B):
            for h in range(att.shape[1]):
                counts.append({'slide_id': names[b], 'true_class':int(y[b]),
                               'attention_head':h, 'n_patches':int(aux['n_patches']),
                               'n_selected':int(aux['selection_count'][b,h]),
                               'selected_fraction':float(aux['selection_count'][b,h]/aux['n_patches'])})
        all_y.extend(y.cpu().tolist()); all_p.extend(probs.cpu().tolist()); slides.extend(names)
    yy=np.asarray(all_y,dtype=int); pp=np.asarray(all_p,dtype=float)
    save_predictions(yy,pp,slides,directory,split)
    pd.DataFrame(counts).to_csv(directory/f'{split}_retention_statistics.csv',index=False)
    return yy,pp

def plot_ablation_validation(stage, folder, history_df, y, p, best_epoch):
    fig,ax=plt.subplots(figsize=(7,4))
    ax.plot(history_df.epoch+1,history_df.train_loss,label='Training objective')
    ax.plot(history_df.epoch+1,history_df.val_loss,label='Validation raw CE')
    ax.axvline(best_epoch+1,color='gray',linestyle='--',label='Selected epoch')
    ax.set(xlabel='Epoch',ylabel='Loss',title=f'{DATASET} {stage}: learning curves')
    ax.legend(); fig.tight_layout(); fig.savefig(folder/'learning_curves.png',dpi=150);plt.close(fig)
    cm=confusion_matrix(y,p.argmax(-1),labels=CLASS_IDS)
    pd.DataFrame(cm,index=CLASS_IDS,columns=CLASS_IDS).to_csv(folder/'validation_confusion_matrix.csv')
    fig,ax=plt.subplots(figsize=(5.5,4.6))
    ConfusionMatrixDisplay(cm,display_labels=CLASS_IDS).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'{DATASET} {stage} validation')
    fig.tight_layout();fig.savefig(folder/'validation_confusion_matrix.png',dpi=150);plt.close(fig)
    fig,ax=plt.subplots(figsize=(6,4.5))
    for c in CLASS_IDS:
        b=(y==c).astype(int)
        if len(np.unique(b))==2:
            fpr,tpr,_=roc_curve(b,p[:,c]); area=roc_auc_score(b,p[:,c]); ax.plot(fpr,tpr,label=f'{c}: {area:.3f}')
    ax.plot([0,1],[0,1],'--',color='gray',lw=1)
    ax.set(xlabel='FPR',ylabel='TPR',title=f'{DATASET} {stage}: validation ROC')
    ax.legend(fontsize=8);fig.tight_layout();fig.savefig(folder/'validation_roc.png',dpi=150);plt.close(fig)
    stats=folder/'validation_retention_statistics.csv'
    if stats.exists():
        stats_df=pd.read_csv(stats)
        if not stats_df.empty:
            fig,ax=plt.subplots(figsize=(6,4))
            ax.hist(stats_df.selected_fraction,bins=20)
            ax.set(xlabel='Fraction of patches selected',ylabel='Class-head observations',
                   title=f'{DATASET} {stage}: evidence retention')
            fig.tight_layout();fig.savefig(folder/'retained_fraction_histogram.png',dpi=150);plt.close(fig)

In [10]:
# Single notebook: new robust trainer. Same model, optimizer and validation score as the original notebook.
# Adds: live per-epoch progress file; atomic best/last checkpoints; resume after a completed epoch.
def _save_atomic_torch(obj, path):
    tmp = Path(str(path)+'.tmp')
    torch.save(obj, tmp)
    os.replace(tmp,path)

def _rng_snapshot():
    return {'torch':torch.get_rng_state(), 'numpy':np.random.get_state(),
            'python':random.getstate(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _restore_rng(s):
    torch.set_rng_state(s['torch']);np.random.set_state(s['numpy']);random.setstate(s['python'])
    if s.get('cuda') is not None and torch.cuda.is_available():torch.cuda.set_rng_state_all(s['cuda'])

def train_ablation(spec):
    stage=spec.name
    folder=stage_dir(stage)
    completed=folder/'COMPLETED.json'
    expected=to_python(asdict(spec))
    if SKIP_COMPLETED_ABLATIONS and completed.is_file():
        try:
            info=json.loads(completed.read_text(encoding='utf-8'))
            if (info.get('spec')==expected and info.get('n_epochs')==ABLATION_EPOCHS
                and info.get('feature_path')==str(Path(conf.data_dir).resolve())):
                progress(f'{DATASET}/{stage}: SKIP (already completed)')
                return info
        except (ValueError,KeyError,TypeError): pass
    seed_everything(int(conf.seed)+spec.seed_offset)
    net=AblationABMIL(conf,spec).to(device)
    opt=torch.optim.AdamW(net.parameters(),lr=conf.lr,weight_decay=conf.wd)
    objective=(LogitAdjustedCrossEntropy(TRAIN_COUNTS,spec.tau).to(device)
               if 'L' in spec.enabled else nn.CrossEntropyLoss())
    best_score=-float('inf');best_epoch=-1;best_state=None;hist=[];first_ep=0
    latest=folder/'last_epoch.pt'
    if RESUME_INCOMPLETE_STAGES and latest.exists():
        ck=torch.load(latest,map_location='cpu',weights_only=False)
        if (ck.get('spec')==expected and ck.get('n_epochs')==ABLATION_EPOCHS
            and ck.get('feature_path')==str(Path(conf.data_dir).resolve())):
            net.load_state_dict(ck['model_state_dict']); opt.load_state_dict(ck['optimizer_state_dict'])
            best_score=ck['best_score'];best_epoch=ck['best_epoch']
            best_state=ck['best_state'];hist=ck['history'];first_ep=ck['completed_epochs']
            _restore_rng(ck['rng'])
            progress(f'{DATASET}/{stage}: RESUME at epoch {first_ep+1}/{ABLATION_EPOCHS}')
    total_start=time.perf_counter()
    progress(f'{DATASET}/{stage}: START, epochs {first_ep+1}–{ABLATION_EPOCHS}, patches as stored, device={device}')
    for ep in range(first_ep,ABLATION_EPOCHS):
        start=time.perf_counter(); net.train();sum_loss=0.;sum_right=0;n=0
        for batch in train_loader:
            x=batch['input'].to(device,dtype=torch.float32)
            y=batch['label'].to(device,dtype=torch.long).reshape(-1)
            opt.zero_grad(set_to_none=True)
            logits,aux=net(x,return_aux=True)
            loss=objective(logits,y)
            if 'R' in spec.enabled:
                loss=loss+spec.proto_lambda*aux_prototype_loss(net,aux,y,spec.temperature)
            loss.backward();opt.step()
            sum_loss+=float(loss.item())*len(y)
            sum_right+=int((logits.argmax(-1)==y).sum().item());n+=len(y)
        val=evaluate(net,nn.CrossEntropyLoss(),val_loader,device,conf)
        auc=float(val['auc']);f1=float(val['f1'])
        score=f1+(auc if np.isfinite(auc) else 0.)
        if score>best_score:
            best_score=score;best_epoch=ep
            best_state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
        row={'epoch':ep,'train_loss':sum_loss/max(1,n),'train_acc':100*sum_right/max(1,n),
             'val_loss':float(val['loss']),'val_acc':float(val['acc']),
             'val_f1':f1,'val_auc':auc,'epoch_seconds':time.perf_counter()-start}
        hist.append(row)
        pd.DataFrame(hist).to_csv(folder/'history_partial.csv',index=False)
        progress(f'{DATASET}/{stage}: epoch {ep+1}/{ABLATION_EPOCHS} '
                 f'val_F1={f1:.4f} val_AUC={auc:.4f} '
                 f'epoch_s={row["epoch_seconds"]:.1f} best={best_epoch+1}')
        # Writing an epoch boundary checkpoint supports resuming interrupted overnight runs.
        _save_atomic_torch({'spec':expected,'n_epochs':ABLATION_EPOCHS,
            'feature_path':str(Path(conf.data_dir).resolve()),
            'model_state_dict':net.state_dict(),'optimizer_state_dict':opt.state_dict(),
            'best_state':best_state,'best_score':best_score,'best_epoch':best_epoch,
            'completed_epochs':ep+1,'history':hist,'rng':_rng_snapshot()},latest)
    if best_state is None: raise RuntimeError('No validation checkpoint selected: '+stage)
    net.load_state_dict(best_state)
    elapsed=time.perf_counter()-total_start
    h=pd.DataFrame(hist);h.to_csv(folder/'history.csv',index=False)
    torch.save({'model_state_dict':best_state,'spec':expected,'best_epoch':best_epoch,
                'dataset':DATASET,'seed':int(conf.seed)+spec.seed_offset},folder/'best_model.pt')
    yy,pp=collect_ablation_details(net,val_loader,folder,split='validation')
    valid_report=prediction_metrics(yy,pp)
    (folder/'validation_classification_report.txt').write_text(
        classification_report(yy,pp.argmax(-1),labels=CLASS_IDS,zero_division=0),encoding='utf-8')
    plot_ablation_validation(stage,folder,h,yy,pp,best_epoch)
    metadata={'dataset':DATASET,'stage':stage,'spec':expected,'n_epochs':ABLATION_EPOCHS,
              'best_epoch_1based':best_epoch+1,'wall_seconds_current_session':elapsed,
              'wall_seconds_completed_epochs':sum(r['epoch_seconds'] for r in hist),
              'n_parameters':sum(p.numel() for p in net.parameters()),
              'validation':valid_report,'held_out_test':None,
              'note':'New variants selected using VALIDATION; test not inspected by default.'}
    if TEST_NEW_ABLATIONS:
        yt,pt=collect_ablation_details(net,test_loader,folder,split='test')
        metadata['held_out_test']=prediction_metrics(yt,pt)
        (folder/'test_classification_report.txt').write_text(
            classification_report(yt,pt.argmax(-1),labels=CLASS_IDS,zero_division=0),encoding='utf-8')
    (folder/'metrics.json').write_text(json.dumps(to_python(metadata),indent=2),encoding='utf-8')
    done={'spec':expected,'n_epochs':ABLATION_EPOCHS,
          'feature_path':str(Path(conf.data_dir).resolve()),
          'validation':valid_report,'wall_seconds':sum(r['epoch_seconds'] for r in hist),
          'best_epoch_1based':best_epoch+1}
    completed.write_text(json.dumps(to_python(done),indent=2),encoding='utf-8')
    if latest.exists(): latest.unlink()
    progress(f'{DATASET}/{stage}: COMPLETED, saved {folder.name}')
    del net,opt
    # Avoid forcing full cyclic GC after every stage; doing so can stall large notebooks.
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return done

---
## Dataset C16
This cell loads only C16's original course split/configuration. After initialization, **each experiment is an independent notebook cell**, so executed notebook results are saved between models and progress is written every epoch to `all_datasets_progress.log`.

In [11]:
# DATASET START: C16 — original course config, loader, and train/validation/test splits.
DATASET = 'C16'
CONFIG_PATH=PROJECT_DIR/'config'/CONFIG_FILES[DATASET]
if DATASET in DATASETS:
    conf=load_config(str(CONFIG_PATH))
    seed_everything(int(conf.seed))
    if not Path(conf.data_dir).exists():
        raise FileNotFoundError('HDF5 feature file missing for '+DATASET+': '+str(conf.data_dir))
    train_data,val_data,test_data=build_HDF5_feat_dataset(conf.data_dir,conf)
    conf.D_feat=infer_feature_dim(train_data)
    if int(conf.B)!=1:
        raise ValueError('Original ABMIL ignores padding masks: use the starter batch size B=1')
    batch_size=conf.B;num_workers=int(conf.n_worker)
    pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
    train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    val_loader=DataLoader(val_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    CLASS_IDS=list(range(int(conf.n_class)))
    TRAIN_COUNTS=train_label_counts()
    NUM_EPOCHS=int(conf.train_epoch if EPOCH_OVERRIDE is None else EPOCH_OVERRIDE)
    ABLATION_EPOCHS=NUM_EPOCHS
    progress(f'C16: initialized train={len(train_data)} val={len(val_data)} test={len(test_data)} epochs={ABLATION_EPOCHS}')
    progress(f'C16: train class counts: {TRAIN_COUNTS.tolist()}')
else:
    progress('C16: skipped (not in DATASETS)')

[2026-11-03T04:41:55] C16: initialized train=242 val=27 test=129 epochs=30
[2026-11-03T04:41:55] C16: train class counts: [142, 100]


In [12]:
# C16 / A0 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A0'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [13]:
# C16 / A1 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A1'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [14]:
# C16 / A2 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A2'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [15]:
# C16 / A3 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A3'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [16]:
# C16 / A4 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A4'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [17]:
# C16 / X_ — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [18]:
# C16 / X_H — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_H'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:41:55] C16/X_H: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:42:01] C16/X_H: epoch 1/30 val_F1=0.8138 val_AUC=0.9091 epoch_s=5.6 best=1
[2026-11-03T04:42:06] C16/X_H: epoch 2/30 val_F1=0.8782 val_AUC=1.0000 epoch_s=5.1 best=2
[2026-11-03T04:42:12] C16/X_H: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.8 best=3
[2026-11-03T04:42:17] C16/X_H: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.5 best=3
[2026-11-03T04:42:23] C16/X_H: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.5 best=3
[2026-11-03T04:42:28] C16/X_H: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.4 best=3
[2026-11-03T04:42:33] C16/X_H: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.4 best=3
[2026-11-03T04:42:39] C16/X_H: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=5.4 best=3
[2026-11-03T04:42:45] C16/X_H: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.0 best=3
[2026-11-03T04:42:51] C16/X_H: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.2 best=3
[2026-11-03T04:42

In [19]:
# C16 / X_P — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_P'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:44:58] C16/X_P: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:45:05] C16/X_P: epoch 1/30 val_F1=0.7970 val_AUC=0.9261 epoch_s=6.9 best=1
[2026-11-03T04:45:11] C16/X_P: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=6.6 best=2
[2026-11-03T04:45:18] C16/X_P: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:45:24] C16/X_P: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:45:31] C16/X_P: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:45:37] C16/X_P: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:45:44] C16/X_P: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:45:51] C16/X_P: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:45:57] C16/X_P: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T04:46:04] C16/X_P: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:46

In [20]:
# C16 / X_R — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_R'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:48:16] C16/X_R: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:48:23] C16/X_R: epoch 1/30 val_F1=0.7970 val_AUC=0.9205 epoch_s=6.7 best=1
[2026-11-03T04:48:29] C16/X_R: epoch 2/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:48:36] C16/X_R: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:48:42] C16/X_R: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:48:49] C16/X_R: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=3
[2026-11-03T04:48:56] C16/X_R: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:49:02] C16/X_R: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:49:09] C16/X_R: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:49:16] C16/X_R: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:49:22] C16/X_R: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:49

In [21]:
# C16 / X_S — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_S'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:51:34] C16/X_S: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:51:40] C16/X_S: epoch 1/30 val_F1=0.7500 val_AUC=0.9148 epoch_s=6.6 best=1
[2026-11-03T04:51:47] C16/X_S: epoch 2/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=6.5 best=2
[2026-11-03T04:51:53] C16/X_S: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52:00] C16/X_S: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52:06] C16/X_S: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52:13] C16/X_S: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52:20] C16/X_S: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52:26] C16/X_S: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.4 best=3
[2026-11-03T04:52:32] C16/X_S: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.4 best=3
[2026-11-03T04:52:39] C16/X_S: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:52

In [22]:
# C16 / X_C — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_C'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:54:50] C16/X_C: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:54:57] C16/X_C: epoch 1/30 val_F1=0.9206 val_AUC=0.9659 epoch_s=7.1 best=1
[2026-11-03T04:55:04] C16/X_C: epoch 2/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=2
[2026-11-03T04:55:11] C16/X_C: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:55:18] C16/X_C: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=2
[2026-11-03T04:55:24] C16/X_C: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:55:31] C16/X_C: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=2
[2026-11-03T04:55:38] C16/X_C: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:55:45] C16/X_C: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=2
[2026-11-03T04:55:51] C16/X_C: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:55:58] C16/X_C: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=2
[2026-11-03T04:56

In [23]:
# C16 / X_HP — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T04:58:15] C16/X_HP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T04:58:22] C16/X_HP: epoch 1/30 val_F1=0.6769 val_AUC=0.8466 epoch_s=6.8 best=1
[2026-11-03T04:58:28] C16/X_HP: epoch 2/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=6.5 best=2
[2026-11-03T04:58:35] C16/X_HP: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T04:58:41] C16/X_HP: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:58:48] C16/X_HP: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:58:55] C16/X_HP: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=3
[2026-11-03T04:59:01] C16/X_HP: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T04:59:08] C16/X_HP: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:59:14] C16/X_HP: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T04:59:21] C16/X_HP: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-

In [24]:
# C16 / X_HR — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:01:33] C16/X_HR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:01:40] C16/X_HR: epoch 1/30 val_F1=0.6769 val_AUC=0.8466 epoch_s=6.7 best=1
[2026-11-03T05:01:46] C16/X_HR: epoch 2/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=6.6 best=2
[2026-11-03T05:01:53] C16/X_HR: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:02:00] C16/X_HR: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T05:02:06] C16/X_HR: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:02:13] C16/X_HR: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:02:20] C16/X_HR: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T05:02:26] C16/X_HR: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T05:02:33] C16/X_HR: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T05:02:40] C16/X_HR: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.8 best=3
[2026-

In [25]:
# C16 / X_PR — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:04:55] C16/X_PR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:05:02] C16/X_PR: epoch 1/30 val_F1=0.7970 val_AUC=0.9261 epoch_s=7.3 best=1
[2026-11-03T05:05:09] C16/X_PR: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.1 best=2
[2026-11-03T05:05:16] C16/X_PR: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:24] C16/X_PR: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:31] C16/X_PR: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:38] C16/X_PR: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:45] C16/X_PR: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:52] C16/X_PR: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:05:59] C16/X_PR: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.0 best=3
[2026-11-03T05:06:06] C16/X_PR: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-

In [26]:
# C16 / X_LHP — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:08:29] C16/X_LHP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:08:36] C16/X_LHP: epoch 1/30 val_F1=0.7336 val_AUC=0.8523 epoch_s=6.8 best=1
[2026-11-03T05:08:43] C16/X_LHP: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=6.6 best=2
[2026-11-03T05:08:49] C16/X_LHP: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T05:08:56] C16/X_LHP: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:09:02] C16/X_LHP: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.5 best=3
[2026-11-03T05:09:09] C16/X_LHP: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.9 best=3
[2026-11-03T05:09:16] C16/X_LHP: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 best=3
[2026-11-03T05:09:22] C16/X_LHP: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:09:29] C16/X_LHP: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.6 best=3
[2026-11-03T05:09:36] C16/X_LHP: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=6.7 be

In [27]:
# C16 / X_LHPR — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:11:50] C16/X_LHPR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:11:57] C16/X_LHPR: epoch 1/30 val_F1=0.7336 val_AUC=0.8466 epoch_s=7.2 best=1
[2026-11-03T05:12:04] C16/X_LHPR: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.2 best=2
[2026-11-03T05:12:11] C16/X_LHPR: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:12:18] C16/X_LHPR: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.2 best=3
[2026-11-03T05:12:26] C16/X_LHPR: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.2 best=3
[2026-11-03T05:12:33] C16/X_LHPR: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:12:40] C16/X_LHPR: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:12:47] C16/X_LHPR: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.0 best=3
[2026-11-03T05:12:54] C16/X_LHPR: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.1 best=3
[2026-11-03T05:13:01] C16/X_LHPR: epoch 10/30 val_F1=1.0000 val_AUC=1.0000 epo

In [28]:
# C16 / X_LHPRS_mass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:15:26] C16/X_LHPRS_mass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:15:34] C16/X_LHPRS_mass80: epoch 1/30 val_F1=0.7336 val_AUC=0.8466 epoch_s=7.7 best=1
[2026-11-03T05:15:41] C16/X_LHPRS_mass80: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.5 best=2
[2026-11-03T05:15:49] C16/X_LHPRS_mass80: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:15:56] C16/X_LHPRS_mass80: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:16:03] C16/X_LHPRS_mass80: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:16:11] C16/X_LHPRS_mass80: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:16:18] C16/X_LHPRS_mass80: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:16:26] C16/X_LHPRS_mass80: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:16:33] C16/X_LHPRS_mass80: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=

In [29]:
# C16 / X_LHPRS_top20 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top20'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:19:11] C16/X_LHPRS_top20: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:19:19] C16/X_LHPRS_top20: epoch 1/30 val_F1=0.7336 val_AUC=0.8523 epoch_s=7.9 best=1
[2026-11-03T05:19:26] C16/X_LHPRS_top20: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.4 best=2
[2026-11-03T05:19:34] C16/X_LHPRS_top20: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:19:41] C16/X_LHPRS_top20: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:19:49] C16/X_LHPRS_top20: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:19:56] C16/X_LHPRS_top20: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:20:04] C16/X_LHPRS_top20: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:20:11] C16/X_LHPRS_top20: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:20:19] C16/X_LHPRS_top20: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11

In [30]:
# C16 / X_LHPRS_top80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:22:57] C16/X_LHPRS_top80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:23:05] C16/X_LHPRS_top80: epoch 1/30 val_F1=0.7336 val_AUC=0.8466 epoch_s=7.6 best=1
[2026-11-03T05:23:12] C16/X_LHPRS_top80: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.5 best=2
[2026-11-03T05:23:20] C16/X_LHPRS_top80: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:23:27] C16/X_LHPRS_top80: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:23:35] C16/X_LHPRS_top80: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:23:42] C16/X_LHPRS_top80: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:23:50] C16/X_LHPRS_top80: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:23:57] C16/X_LHPRS_top80: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:24:05] C16/X_LHPRS_top80: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11

In [31]:
# C16 / X_LHPRS_top16 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top16'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:26:43] C16/X_LHPRS_top16: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:26:51] C16/X_LHPRS_top16: epoch 1/30 val_F1=0.7336 val_AUC=0.8636 epoch_s=7.6 best=1
[2026-11-03T05:26:58] C16/X_LHPRS_top16: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.4 best=2
[2026-11-03T05:27:06] C16/X_LHPRS_top16: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:27:13] C16/X_LHPRS_top16: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:27:21] C16/X_LHPRS_top16: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:27:28] C16/X_LHPRS_top16: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:27:36] C16/X_LHPRS_top16: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:27:43] C16/X_LHPRS_top16: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:27:51] C16/X_LHPRS_top16: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11

In [32]:
# C16 / X_LHPRS_gatedmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_gatedmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:30:30] C16/X_LHPRS_gatedmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:30:37] C16/X_LHPRS_gatedmass80: epoch 1/30 val_F1=0.7336 val_AUC=0.8466 epoch_s=7.7 best=1
[2026-11-03T05:30:45] C16/X_LHPRS_gatedmass80: epoch 2/30 val_F1=0.9610 val_AUC=1.0000 epoch_s=7.4 best=2
[2026-11-03T05:30:52] C16/X_LHPRS_gatedmass80: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:31:00] C16/X_LHPRS_gatedmass80: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:31:07] C16/X_LHPRS_gatedmass80: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:31:15] C16/X_LHPRS_gatedmass80: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:31:22] C16/X_LHPRS_gatedmass80: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.4 best=3
[2026-11-03T05:31:30] C16/X_LHPRS_gatedmass80: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.7 best=3
[2026-11-03T05:31:38] C16/X_LHPRS_gatedmass80: epoch 9

In [33]:
# C16 / X_LHPRS_poolmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_poolmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:34:17] C16/X_LHPRS_poolmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:34:24] C16/X_LHPRS_poolmass80: epoch 1/30 val_F1=0.7336 val_AUC=0.8636 epoch_s=7.7 best=1
[2026-11-03T05:34:32] C16/X_LHPRS_poolmass80: epoch 2/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=7.6 best=2
[2026-11-03T05:34:39] C16/X_LHPRS_poolmass80: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.6 best=3
[2026-11-03T05:34:47] C16/X_LHPRS_poolmass80: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.7 best=3
[2026-11-03T05:34:55] C16/X_LHPRS_poolmass80: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.6 best=3
[2026-11-03T05:35:02] C16/X_LHPRS_poolmass80: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:35:10] C16/X_LHPRS_poolmass80: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:35:17] C16/X_LHPRS_poolmass80: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.5 best=3
[2026-11-03T05:35:25] C16/X_LHPRS_poolmass80: epoch 9/30 val_F1

In [34]:
# C16 / X_LHPRC_low — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_low'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:38:05] C16/X_LHPRC_low: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:38:13] C16/X_LHPRC_low: epoch 1/30 val_F1=0.9206 val_AUC=1.0000 epoch_s=8.0 best=1
[2026-11-03T05:38:20] C16/X_LHPRC_low: epoch 2/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:38:28] C16/X_LHPRC_low: epoch 3/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:38:36] C16/X_LHPRC_low: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:38:44] C16/X_LHPRC_low: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.9 best=2
[2026-11-03T05:38:52] C16/X_LHPRC_low: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.9 best=2
[2026-11-03T05:39:00] C16/X_LHPRC_low: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:39:08] C16/X_LHPRC_low: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.9 best=2
[2026-11-03T05:39:16] C16/X_LHPRC_low: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:39:23] C16/X_

In [35]:
# C16 / X_LHPRC_before — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_before'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:42:03] C16/X_LHPRC_before: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:42:11] C16/X_LHPRC_before: epoch 1/30 val_F1=0.7699 val_AUC=0.8977 epoch_s=8.1 best=1
[2026-11-03T05:42:19] C16/X_LHPRC_before: epoch 2/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:42:26] C16/X_LHPRC_before: epoch 3/30 val_F1=0.9250 val_AUC=1.0000 epoch_s=7.6 best=2
[2026-11-03T05:42:34] C16/X_LHPRC_before: epoch 4/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.6 best=2
[2026-11-03T05:42:42] C16/X_LHPRC_before: epoch 5/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=8.1 best=2
[2026-11-03T05:42:50] C16/X_LHPRC_before: epoch 6/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:42:58] C16/X_LHPRC_before: epoch 7/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:43:06] C16/X_LHPRC_before: epoch 8/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.8 best=2
[2026-11-03T05:43:14] C16/X_LHPRC_before: epoch 9/30 val_F1=1.0000 val_AUC=1.0000 epoch_s=7.7 best=

In [36]:
# C16 / X_LHPRS_mass90 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass90'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [37]:
# C16 / X_LHPR_lam001 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam001'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [38]:
# C16 / X_LHPR_lam010 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam010'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [39]:
# C16 / X_LHPR_seed43 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed43'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [40]:
# C16 / X_LHPR_seed44 — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed44'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [41]:
# C16 / X_L — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_L'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [42]:
# C16 / X_LH — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LH'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [43]:
# C16 / X_LP — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [44]:
# C16 / X_LR — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [45]:
# C16 / X_LS — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [46]:
# C16 / X_LC — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [47]:
# C16 / X_HS — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [48]:
# C16 / X_HC — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [49]:
# C16 / X_PS — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [50]:
# C16 / X_PC — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [51]:
# C16 / X_RS — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [52]:
# C16 / X_RC — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [53]:
# C16 / X_SC — only runs if this stage is selected in CELL 1.
if DATASET == 'C16' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_SC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [54]:
# C16 — validation comparison, saved after all selected stages.
if DATASET == 'C16' and DATASET in DATASETS:
    folder=RESULTS_ROOT/f'results_{DATASET}_ablation_comparison'
    folder.mkdir(parents=True,exist_ok=True)
    rows=[]
    for spec in STAGE_SPECS:
        path=stage_dir(spec.name)/'COMPLETED.json'
        if not path.exists():continue
        info=json.loads(path.read_text(encoding='utf-8'))
        if (info.get('spec')!=to_python(asdict(spec)) or
            info.get('n_epochs')!=ABLATION_EPOCHS):continue
        v=info['validation']
        rows.append({'stage':spec.name,'switches':'+'.join(spec.enabled),
           'val_accuracy':v['accuracy'],'val_macro_f1':v['macro_f1'],
           'val_macro_ovr_auc':v['macro_ovr_auc'],
           'best_epoch':info['best_epoch_1based'],
           'minutes':info['wall_seconds']/60})
    if rows:
        df=pd.DataFrame(rows).sort_values('val_macro_f1',ascending=False)
        df.to_csv(folder/'all_validation_results.csv',index=False)
        fig,ax=plt.subplots(figsize=(max(8,len(df)*.42),4))
        ax.bar(df.stage,df.val_macro_f1);ax.tick_params(axis='x',rotation=75)
        ax.set(title=f'C16: new ablation validation macro-F1',ylabel='Macro-F1')
        fig.tight_layout();fig.savefig(folder/'validation_macro_f1.png',dpi=140);plt.close(fig)
        print(df.to_string(index=False))
        progress('C16: comparisons saved to '+str(folder))
    else:
        progress('C16: no selected stages completed')
    del train_loader,val_loader,test_loader,train_data,val_data,test_data
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

              stage  switches  val_accuracy  val_macro_f1  val_macro_ovr_auc  best_epoch  minutes
                X_H         H         100.0           1.0                1.0           3 3.005063
                X_P         P         100.0           1.0                1.0           3 3.272121
                X_R         R         100.0           1.0                1.0           3 3.268372
                X_S         S         100.0           1.0                1.0           3 3.238296
                X_C         C         100.0           1.0                1.0           2 3.380146
               X_HP       H+P         100.0           1.0                1.0           3 3.272299
               X_HR       H+R         100.0           1.0                1.0           3 3.332418
               X_PR       P+R         100.0           1.0                1.0           3 3.539189
              X_LHP     L+H+P         100.0           1.0                1.0           3 3.310924
             X_LHPR 

---
## Dataset C17
This cell loads only C17's original course split/configuration. After initialization, **each experiment is an independent notebook cell**, so executed notebook results are saved between models and progress is written every epoch to `all_datasets_progress.log`.

In [55]:
# DATASET START: C17 — original course config, loader, and train/validation/test splits.
DATASET = 'C17'
CONFIG_PATH=PROJECT_DIR/'config'/CONFIG_FILES[DATASET]
if DATASET in DATASETS:
    conf=load_config(str(CONFIG_PATH))
    seed_everything(int(conf.seed))
    if not Path(conf.data_dir).exists():
        raise FileNotFoundError('HDF5 feature file missing for '+DATASET+': '+str(conf.data_dir))
    train_data,val_data,test_data=build_HDF5_feat_dataset(conf.data_dir,conf)
    conf.D_feat=infer_feature_dim(train_data)
    if int(conf.B)!=1:
        raise ValueError('Original ABMIL ignores padding masks: use the starter batch size B=1')
    batch_size=conf.B;num_workers=int(conf.n_worker)
    pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
    train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    val_loader=DataLoader(val_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    CLASS_IDS=list(range(int(conf.n_class)))
    TRAIN_COUNTS=train_label_counts()
    NUM_EPOCHS=int(conf.train_epoch if EPOCH_OVERRIDE is None else EPOCH_OVERRIDE)
    ABLATION_EPOCHS=NUM_EPOCHS
    progress(f'C17: initialized train={len(train_data)} val={len(val_data)} test={len(test_data)} epochs={ABLATION_EPOCHS}')
    progress(f'C17: train class counts: {TRAIN_COUNTS.tolist()}')
else:
    progress('C17: skipped (not in DATASETS)')

[2026-11-03T05:45:59] C17: initialized train=269 val=30 test=200 epochs=30
[2026-11-03T05:45:59] C17: train class counts: [177, 18, 37, 37]


In [56]:
# C17 / A0 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A0'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [57]:
# C17 / A1 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A1'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [58]:
# C17 / A2 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A2'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [59]:
# C17 / A3 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A3'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [60]:
# C17 / A4 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A4'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [61]:
# C17 / X_ — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [62]:
# C17 / X_H — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_H'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:45:59] C17/X_H: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:46:13] C17/X_H: epoch 1/30 val_F1=0.2000 val_AUC=0.6067 epoch_s=14.0 best=1
[2026-11-03T05:46:24] C17/X_H: epoch 2/30 val_F1=0.2000 val_AUC=0.5746 epoch_s=10.8 best=1
[2026-11-03T05:46:35] C17/X_H: epoch 3/30 val_F1=0.3939 val_AUC=0.7808 epoch_s=11.1 best=3
[2026-11-03T05:46:46] C17/X_H: epoch 4/30 val_F1=0.5802 val_AUC=0.8521 epoch_s=11.2 best=4
[2026-11-03T05:46:57] C17/X_H: epoch 5/30 val_F1=0.6399 val_AUC=0.8536 epoch_s=10.9 best=5
[2026-11-03T05:47:09] C17/X_H: epoch 6/30 val_F1=0.6048 val_AUC=0.8491 epoch_s=11.6 best=5
[2026-11-03T05:47:20] C17/X_H: epoch 7/30 val_F1=0.6399 val_AUC=0.8989 epoch_s=11.1 best=7
[2026-11-03T05:47:31] C17/X_H: epoch 8/30 val_F1=0.7435 val_AUC=0.8876 epoch_s=11.5 best=8
[2026-11-03T05:47:43] C17/X_H: epoch 9/30 val_F1=0.6343 val_AUC=0.8833 epoch_s=11.3 best=8
[2026-11-03T05:47:54] C17/X_H: epoch 10/30 val_F1=0.7683 val_AUC=0.8859 epoch_s=10.8 best=10
[2026-

In [63]:
# C17 / X_P — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_P'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:51:39] C17/X_P: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:51:50] C17/X_P: epoch 1/30 val_F1=0.2000 val_AUC=0.5340 epoch_s=11.3 best=1
[2026-11-03T05:52:02] C17/X_P: epoch 2/30 val_F1=0.2000 val_AUC=0.5743 epoch_s=11.6 best=2
[2026-11-03T05:52:13] C17/X_P: epoch 3/30 val_F1=0.3128 val_AUC=0.7737 epoch_s=11.6 best=3
[2026-11-03T05:52:25] C17/X_P: epoch 4/30 val_F1=0.5495 val_AUC=0.8888 epoch_s=11.3 best=4
[2026-11-03T05:52:36] C17/X_P: epoch 5/30 val_F1=0.5495 val_AUC=0.8825 epoch_s=11.6 best=4
[2026-11-03T05:52:48] C17/X_P: epoch 6/30 val_F1=0.6214 val_AUC=0.8689 epoch_s=11.6 best=6
[2026-11-03T05:52:59] C17/X_P: epoch 7/30 val_F1=0.6139 val_AUC=0.8870 epoch_s=11.5 best=7
[2026-11-03T05:53:11] C17/X_P: epoch 8/30 val_F1=0.4981 val_AUC=0.8390 epoch_s=11.6 best=7
[2026-11-03T05:53:22] C17/X_P: epoch 9/30 val_F1=0.6746 val_AUC=0.8745 epoch_s=11.2 best=9
[2026-11-03T05:53:34] C17/X_P: epoch 10/30 val_F1=0.5625 val_AUC=0.8809 epoch_s=11.5 best=9
[2026-1

In [64]:
# C17 / X_R — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_R'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T05:57:26] C17/X_R: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T05:57:37] C17/X_R: epoch 1/30 val_F1=0.2000 val_AUC=0.5324 epoch_s=11.3 best=1
[2026-11-03T05:57:48] C17/X_R: epoch 2/30 val_F1=0.2000 val_AUC=0.5590 epoch_s=11.4 best=2
[2026-11-03T05:58:00] C17/X_R: epoch 3/30 val_F1=0.3128 val_AUC=0.7917 epoch_s=11.6 best=3
[2026-11-03T05:58:11] C17/X_R: epoch 4/30 val_F1=0.5495 val_AUC=0.8945 epoch_s=11.1 best=4
[2026-11-03T05:58:22] C17/X_R: epoch 5/30 val_F1=0.6214 val_AUC=0.8752 epoch_s=11.3 best=5
[2026-11-03T05:58:34] C17/X_R: epoch 6/30 val_F1=0.6214 val_AUC=0.8614 epoch_s=11.2 best=5
[2026-11-03T05:58:45] C17/X_R: epoch 7/30 val_F1=0.5902 val_AUC=0.8882 epoch_s=11.2 best=5
[2026-11-03T05:58:56] C17/X_R: epoch 8/30 val_F1=0.4981 val_AUC=0.8338 epoch_s=11.1 best=5
[2026-11-03T05:59:08] C17/X_R: epoch 9/30 val_F1=0.6214 val_AUC=0.8535 epoch_s=11.6 best=5
[2026-11-03T05:59:19] C17/X_R: epoch 10/30 val_F1=0.5859 val_AUC=0.8753 epoch_s=11.0 best=5
[2026-1

In [65]:
# C17 / X_S — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_S'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:03:06] C17/X_S: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:03:17] C17/X_S: epoch 1/30 val_F1=0.2000 val_AUC=0.4682 epoch_s=11.1 best=1
[2026-11-03T06:03:29] C17/X_S: epoch 2/30 val_F1=0.2000 val_AUC=0.5636 epoch_s=11.4 best=2
[2026-11-03T06:03:40] C17/X_S: epoch 3/30 val_F1=0.4049 val_AUC=0.8019 epoch_s=11.4 best=3
[2026-11-03T06:03:52] C17/X_S: epoch 4/30 val_F1=0.5326 val_AUC=0.8794 epoch_s=11.6 best=4
[2026-11-03T06:04:03] C17/X_S: epoch 5/30 val_F1=0.5004 val_AUC=0.8920 epoch_s=11.4 best=4
[2026-11-03T06:04:15] C17/X_S: epoch 6/30 val_F1=0.6214 val_AUC=0.8779 epoch_s=11.6 best=6
[2026-11-03T06:04:27] C17/X_S: epoch 7/30 val_F1=0.6214 val_AUC=0.8683 epoch_s=11.6 best=6
[2026-11-03T06:04:38] C17/X_S: epoch 8/30 val_F1=0.5004 val_AUC=0.8796 epoch_s=11.7 best=6
[2026-11-03T06:04:50] C17/X_S: epoch 9/30 val_F1=0.6214 val_AUC=0.8644 epoch_s=11.7 best=6
[2026-11-03T06:05:02] C17/X_S: epoch 10/30 val_F1=0.5629 val_AUC=0.8295 epoch_s=11.5 best=6
[2026-1

In [66]:
# C17 / X_C — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_C'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:08:51] C17/X_C: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:09:02] C17/X_C: epoch 1/30 val_F1=0.2000 val_AUC=0.5891 epoch_s=11.2 best=1
[2026-11-03T06:09:14] C17/X_C: epoch 2/30 val_F1=0.4144 val_AUC=0.8123 epoch_s=11.5 best=2
[2026-11-03T06:09:26] C17/X_C: epoch 3/30 val_F1=0.5381 val_AUC=0.8726 epoch_s=11.6 best=3
[2026-11-03T06:09:37] C17/X_C: epoch 4/30 val_F1=0.6032 val_AUC=0.8982 epoch_s=11.8 best=4
[2026-11-03T06:09:49] C17/X_C: epoch 5/30 val_F1=0.7161 val_AUC=0.9005 epoch_s=11.6 best=5
[2026-11-03T06:10:01] C17/X_C: epoch 6/30 val_F1=0.6582 val_AUC=0.9143 epoch_s=11.8 best=5
[2026-11-03T06:10:13] C17/X_C: epoch 7/30 val_F1=0.8471 val_AUC=0.9314 epoch_s=11.8 best=7
[2026-11-03T06:10:24] C17/X_C: epoch 8/30 val_F1=0.7022 val_AUC=0.9198 epoch_s=11.4 best=7
[2026-11-03T06:10:36] C17/X_C: epoch 9/30 val_F1=0.7772 val_AUC=0.9156 epoch_s=11.7 best=7
[2026-11-03T06:10:47] C17/X_C: epoch 10/30 val_F1=0.8471 val_AUC=0.8896 epoch_s=11.4 best=7
[2026-1

In [67]:
# C17 / X_HP — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:14:39] C17/X_HP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:14:51] C17/X_HP: epoch 1/30 val_F1=0.2000 val_AUC=0.5907 epoch_s=11.3 best=1
[2026-11-03T06:15:02] C17/X_HP: epoch 2/30 val_F1=0.2000 val_AUC=0.6501 epoch_s=11.5 best=2
[2026-11-03T06:15:14] C17/X_HP: epoch 3/30 val_F1=0.4148 val_AUC=0.7998 epoch_s=11.3 best=3
[2026-11-03T06:15:25] C17/X_HP: epoch 4/30 val_F1=0.5326 val_AUC=0.8438 epoch_s=11.6 best=4
[2026-11-03T06:15:37] C17/X_HP: epoch 5/30 val_F1=0.6968 val_AUC=0.8344 epoch_s=11.4 best=5
[2026-11-03T06:15:48] C17/X_HP: epoch 6/30 val_F1=0.6756 val_AUC=0.8741 epoch_s=11.6 best=6
[2026-11-03T06:16:00] C17/X_HP: epoch 7/30 val_F1=0.6968 val_AUC=0.9034 epoch_s=11.7 best=7
[2026-11-03T06:16:11] C17/X_HP: epoch 8/30 val_F1=0.6548 val_AUC=0.8889 epoch_s=11.3 best=7
[2026-11-03T06:16:23] C17/X_HP: epoch 9/30 val_F1=0.8000 val_AUC=0.9112 epoch_s=11.3 best=9
[2026-11-03T06:16:34] C17/X_HP: epoch 10/30 val_F1=0.6960 val_AUC=0.9014 epoch_s=11.6 bes

In [68]:
# C17 / X_HR — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:20:28] C17/X_HR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:20:39] C17/X_HR: epoch 1/30 val_F1=0.2000 val_AUC=0.5886 epoch_s=11.5 best=1
[2026-11-03T06:20:51] C17/X_HR: epoch 2/30 val_F1=0.2000 val_AUC=0.6299 epoch_s=11.4 best=2
[2026-11-03T06:21:02] C17/X_HR: epoch 3/30 val_F1=0.4148 val_AUC=0.7926 epoch_s=11.5 best=3
[2026-11-03T06:21:14] C17/X_HR: epoch 4/30 val_F1=0.5326 val_AUC=0.8483 epoch_s=11.4 best=4
[2026-11-03T06:21:25] C17/X_HR: epoch 5/30 val_F1=0.6492 val_AUC=0.8333 epoch_s=11.1 best=5
[2026-11-03T06:21:36] C17/X_HR: epoch 6/30 val_F1=0.6756 val_AUC=0.8700 epoch_s=11.3 best=6
[2026-11-03T06:21:47] C17/X_HR: epoch 7/30 val_F1=0.6310 val_AUC=0.8883 epoch_s=11.2 best=6
[2026-11-03T06:21:59] C17/X_HR: epoch 8/30 val_F1=0.6492 val_AUC=0.8784 epoch_s=11.6 best=6
[2026-11-03T06:22:10] C17/X_HR: epoch 9/30 val_F1=0.8000 val_AUC=0.9014 epoch_s=11.2 best=9
[2026-11-03T06:22:22] C17/X_HR: epoch 10/30 val_F1=0.7024 val_AUC=0.9082 epoch_s=11.4 bes

In [69]:
# C17 / X_PR — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:26:12] C17/X_PR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:26:24] C17/X_PR: epoch 1/30 val_F1=0.2000 val_AUC=0.5287 epoch_s=11.9 best=1
[2026-11-03T06:26:36] C17/X_PR: epoch 2/30 val_F1=0.2000 val_AUC=0.5517 epoch_s=12.1 best=2
[2026-11-03T06:26:48] C17/X_PR: epoch 3/30 val_F1=0.3083 val_AUC=0.7761 epoch_s=12.1 best=3
[2026-11-03T06:27:00] C17/X_PR: epoch 4/30 val_F1=0.5495 val_AUC=0.8836 epoch_s=11.9 best=4
[2026-11-03T06:27:12] C17/X_PR: epoch 5/30 val_F1=0.4981 val_AUC=0.8692 epoch_s=11.8 best=4
[2026-11-03T06:27:23] C17/X_PR: epoch 6/30 val_F1=0.5495 val_AUC=0.8505 epoch_s=11.8 best=4
[2026-11-03T06:27:35] C17/X_PR: epoch 7/30 val_F1=0.5902 val_AUC=0.8927 epoch_s=11.8 best=7
[2026-11-03T06:27:47] C17/X_PR: epoch 8/30 val_F1=0.4981 val_AUC=0.8322 epoch_s=11.8 best=7
[2026-11-03T06:27:59] C17/X_PR: epoch 9/30 val_F1=0.6214 val_AUC=0.8601 epoch_s=11.8 best=7
[2026-11-03T06:28:11] C17/X_PR: epoch 10/30 val_F1=0.5625 val_AUC=0.8847 epoch_s=12.0 bes

In [70]:
# C17 / X_LHP — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:32:11] C17/X_LHP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:32:22] C17/X_LHP: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=11.1 best=1
[2026-11-03T06:32:34] C17/X_LHP: epoch 2/30 val_F1=0.1970 val_AUC=0.6138 epoch_s=11.3 best=1
[2026-11-03T06:32:45] C17/X_LHP: epoch 3/30 val_F1=0.4084 val_AUC=0.7951 epoch_s=11.6 best=3
[2026-11-03T06:32:57] C17/X_LHP: epoch 4/30 val_F1=0.4548 val_AUC=0.8398 epoch_s=11.4 best=4
[2026-11-03T06:33:08] C17/X_LHP: epoch 5/30 val_F1=0.5897 val_AUC=0.8581 epoch_s=11.6 best=5
[2026-11-03T06:33:20] C17/X_LHP: epoch 6/30 val_F1=0.5924 val_AUC=0.9152 epoch_s=11.2 best=6
[2026-11-03T06:33:31] C17/X_LHP: epoch 7/30 val_F1=0.7161 val_AUC=0.9318 epoch_s=11.1 best=7
[2026-11-03T06:33:42] C17/X_LHP: epoch 8/30 val_F1=0.7606 val_AUC=0.9037 epoch_s=11.4 best=8
[2026-11-03T06:33:53] C17/X_LHP: epoch 9/30 val_F1=0.7683 val_AUC=0.9085 epoch_s=11.3 best=9
[2026-11-03T06:34:05] C17/X_LHP: epoch 10/30 val_F1=0.7683 val_AUC=0.8917 epoch

In [71]:
# C17 / X_LHPR — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:37:56] C17/X_LHPR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:38:08] C17/X_LHPR: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=11.9 best=1
[2026-11-03T06:38:20] C17/X_LHPR: epoch 2/30 val_F1=0.1970 val_AUC=0.6005 epoch_s=12.0 best=1
[2026-11-03T06:38:32] C17/X_LHPR: epoch 3/30 val_F1=0.4084 val_AUC=0.7778 epoch_s=12.1 best=3
[2026-11-03T06:38:43] C17/X_LHPR: epoch 4/30 val_F1=0.4548 val_AUC=0.8422 epoch_s=11.7 best=4
[2026-11-03T06:38:55] C17/X_LHPR: epoch 5/30 val_F1=0.5826 val_AUC=0.8734 epoch_s=12.0 best=5
[2026-11-03T06:39:07] C17/X_LHPR: epoch 6/30 val_F1=0.6814 val_AUC=0.8896 epoch_s=11.8 best=6
[2026-11-03T06:39:19] C17/X_LHPR: epoch 7/30 val_F1=0.6756 val_AUC=0.9261 epoch_s=11.8 best=7
[2026-11-03T06:39:31] C17/X_LHPR: epoch 8/30 val_F1=0.7606 val_AUC=0.8947 epoch_s=11.8 best=8
[2026-11-03T06:39:43] C17/X_LHPR: epoch 9/30 val_F1=0.7683 val_AUC=0.9049 epoch_s=12.2 best=9
[2026-11-03T06:39:55] C17/X_LHPR: epoch 10/30 val_F1=0.8018 val_AUC=0

In [72]:
# C17 / X_LHPRS_mass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:43:54] C17/X_LHPRS_mass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:44:07] C17/X_LHPRS_mass80: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=12.0 best=1
[2026-11-03T06:44:18] C17/X_LHPRS_mass80: epoch 2/30 val_F1=0.1970 val_AUC=0.5960 epoch_s=11.9 best=1
[2026-11-03T06:44:30] C17/X_LHPRS_mass80: epoch 3/30 val_F1=0.4084 val_AUC=0.7758 epoch_s=11.9 best=3
[2026-11-03T06:44:42] C17/X_LHPRS_mass80: epoch 4/30 val_F1=0.4548 val_AUC=0.8419 epoch_s=12.0 best=4
[2026-11-03T06:44:54] C17/X_LHPRS_mass80: epoch 5/30 val_F1=0.6492 val_AUC=0.8690 epoch_s=11.8 best=5
[2026-11-03T06:45:06] C17/X_LHPRS_mass80: epoch 6/30 val_F1=0.6756 val_AUC=0.8916 epoch_s=12.1 best=6
[2026-11-03T06:45:18] C17/X_LHPRS_mass80: epoch 7/30 val_F1=0.6756 val_AUC=0.9305 epoch_s=12.0 best=7
[2026-11-03T06:45:31] C17/X_LHPRS_mass80: epoch 8/30 val_F1=0.7606 val_AUC=0.9053 epoch_s=12.1 best=8
[2026-11-03T06:45:43] C17/X_LHPRS_mass80: epoch 9/30 val_F1=0.8000 val_AUC=0.9082 epoch_s=1

In [73]:
# C17 / X_LHPRS_top20 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top20'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:49:57] C17/X_LHPRS_top20: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:50:09] C17/X_LHPRS_top20: epoch 1/30 val_F1=0.1939 val_AUC=0.6232 epoch_s=11.8 best=1
[2026-11-03T06:50:21] C17/X_LHPRS_top20: epoch 2/30 val_F1=0.1970 val_AUC=0.5936 epoch_s=11.9 best=1
[2026-11-03T06:50:33] C17/X_LHPRS_top20: epoch 3/30 val_F1=0.4084 val_AUC=0.7770 epoch_s=12.2 best=3
[2026-11-03T06:50:45] C17/X_LHPRS_top20: epoch 4/30 val_F1=0.4548 val_AUC=0.8395 epoch_s=12.4 best=4
[2026-11-03T06:50:57] C17/X_LHPRS_top20: epoch 5/30 val_F1=0.5897 val_AUC=0.8573 epoch_s=12.1 best=5
[2026-11-03T06:51:09] C17/X_LHPRS_top20: epoch 6/30 val_F1=0.6190 val_AUC=0.8994 epoch_s=12.1 best=6
[2026-11-03T06:51:21] C17/X_LHPRS_top20: epoch 7/30 val_F1=0.6968 val_AUC=0.9294 epoch_s=12.0 best=7
[2026-11-03T06:51:34] C17/X_LHPRS_top20: epoch 8/30 val_F1=0.7606 val_AUC=0.9026 epoch_s=12.0 best=8
[2026-11-03T06:51:46] C17/X_LHPRS_top20: epoch 9/30 val_F1=0.8000 val_AUC=0.9037 epoch_s=12.2 best=9

In [74]:
# C17 / X_LHPRS_top80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T06:56:01] C17/X_LHPRS_top80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T06:56:13] C17/X_LHPRS_top80: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=12.2 best=1
[2026-11-03T06:56:25] C17/X_LHPRS_top80: epoch 2/30 val_F1=0.1970 val_AUC=0.6017 epoch_s=12.1 best=1
[2026-11-03T06:56:37] C17/X_LHPRS_top80: epoch 3/30 val_F1=0.4084 val_AUC=0.7758 epoch_s=12.1 best=3
[2026-11-03T06:56:50] C17/X_LHPRS_top80: epoch 4/30 val_F1=0.4548 val_AUC=0.8443 epoch_s=12.1 best=4
[2026-11-03T06:57:02] C17/X_LHPRS_top80: epoch 5/30 val_F1=0.6492 val_AUC=0.8734 epoch_s=12.2 best=5
[2026-11-03T06:57:14] C17/X_LHPRS_top80: epoch 6/30 val_F1=0.6756 val_AUC=0.8916 epoch_s=12.1 best=6
[2026-11-03T06:57:26] C17/X_LHPRS_top80: epoch 7/30 val_F1=0.6756 val_AUC=0.9305 epoch_s=12.0 best=7
[2026-11-03T06:57:38] C17/X_LHPRS_top80: epoch 8/30 val_F1=0.7606 val_AUC=0.9053 epoch_s=12.0 best=8
[2026-11-03T06:57:50] C17/X_LHPRS_top80: epoch 9/30 val_F1=0.8000 val_AUC=0.9057 epoch_s=12.1 best=9

In [75]:
# C17 / X_LHPRS_top16 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top16'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:02:04] C17/X_LHPRS_top16: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:02:16] C17/X_LHPRS_top16: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=12.0 best=1
[2026-11-03T07:02:28] C17/X_LHPRS_top16: epoch 2/30 val_F1=0.1970 val_AUC=0.5863 epoch_s=12.0 best=1
[2026-11-03T07:02:41] C17/X_LHPRS_top16: epoch 3/30 val_F1=0.4084 val_AUC=0.7718 epoch_s=12.2 best=3
[2026-11-03T07:02:53] C17/X_LHPRS_top16: epoch 4/30 val_F1=0.4548 val_AUC=0.8350 epoch_s=12.2 best=4
[2026-11-03T07:03:05] C17/X_LHPRS_top16: epoch 5/30 val_F1=0.5826 val_AUC=0.8669 epoch_s=12.0 best=5
[2026-11-03T07:03:17] C17/X_LHPRS_top16: epoch 6/30 val_F1=0.6814 val_AUC=0.8851 epoch_s=12.1 best=6
[2026-11-03T07:03:29] C17/X_LHPRS_top16: epoch 7/30 val_F1=0.6968 val_AUC=0.9184 epoch_s=12.1 best=7
[2026-11-03T07:03:41] C17/X_LHPRS_top16: epoch 8/30 val_F1=0.7606 val_AUC=0.8895 epoch_s=12.1 best=8
[2026-11-03T07:03:53] C17/X_LHPRS_top16: epoch 9/30 val_F1=0.7304 val_AUC=0.9025 epoch_s=12.1 best=8

In [76]:
# C17 / X_LHPRS_gatedmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_gatedmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:08:09] C17/X_LHPRS_gatedmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:08:21] C17/X_LHPRS_gatedmass80: epoch 1/30 val_F1=0.1939 val_AUC=0.6289 epoch_s=12.0 best=1
[2026-11-03T07:08:33] C17/X_LHPRS_gatedmass80: epoch 2/30 val_F1=0.1970 val_AUC=0.6029 epoch_s=12.1 best=1
[2026-11-03T07:08:45] C17/X_LHPRS_gatedmass80: epoch 3/30 val_F1=0.4084 val_AUC=0.7758 epoch_s=12.1 best=3
[2026-11-03T07:08:57] C17/X_LHPRS_gatedmass80: epoch 4/30 val_F1=0.4548 val_AUC=0.8419 epoch_s=12.0 best=4
[2026-11-03T07:09:09] C17/X_LHPRS_gatedmass80: epoch 5/30 val_F1=0.6492 val_AUC=0.8666 epoch_s=11.9 best=5
[2026-11-03T07:09:21] C17/X_LHPRS_gatedmass80: epoch 6/30 val_F1=0.6190 val_AUC=0.8916 epoch_s=12.1 best=5
[2026-11-03T07:09:34] C17/X_LHPRS_gatedmass80: epoch 7/30 val_F1=0.6756 val_AUC=0.9281 epoch_s=12.1 best=7
[2026-11-03T07:09:46] C17/X_LHPRS_gatedmass80: epoch 8/30 val_F1=0.7606 val_AUC=0.9053 epoch_s=12.2 best=8
[2026-11-03T07:09:58] C17/X_LHPRS_gatedmass80:

In [77]:
# C17 / X_LHPRS_poolmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_poolmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:14:13] C17/X_LHPRS_poolmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:14:25] C17/X_LHPRS_poolmass80: epoch 1/30 val_F1=0.1939 val_AUC=0.6232 epoch_s=12.1 best=1
[2026-11-03T07:14:38] C17/X_LHPRS_poolmass80: epoch 2/30 val_F1=0.1970 val_AUC=0.6489 epoch_s=12.2 best=2
[2026-11-03T07:14:50] C17/X_LHPRS_poolmass80: epoch 3/30 val_F1=0.4137 val_AUC=0.7976 epoch_s=12.3 best=3
[2026-11-03T07:15:02] C17/X_LHPRS_poolmass80: epoch 4/30 val_F1=0.4548 val_AUC=0.8398 epoch_s=12.1 best=4
[2026-11-03T07:15:14] C17/X_LHPRS_poolmass80: epoch 5/30 val_F1=0.5897 val_AUC=0.8731 epoch_s=12.2 best=5
[2026-11-03T07:15:27] C17/X_LHPRS_poolmass80: epoch 6/30 val_F1=0.5548 val_AUC=0.9084 epoch_s=12.4 best=6
[2026-11-03T07:15:39] C17/X_LHPRS_poolmass80: epoch 7/30 val_F1=0.6968 val_AUC=0.9294 epoch_s=12.2 best=7
[2026-11-03T07:15:51] C17/X_LHPRS_poolmass80: epoch 8/30 val_F1=0.7606 val_AUC=0.9095 epoch_s=12.3 best=8
[2026-11-03T07:16:03] C17/X_LHPRS_poolmass80: epoch 9/3

In [78]:
# C17 / X_LHPRC_low — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_low'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:20:22] C17/X_LHPRC_low: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:20:34] C17/X_LHPRC_low: epoch 1/30 val_F1=0.2000 val_AUC=0.6509 epoch_s=12.2 best=1
[2026-11-03T07:20:46] C17/X_LHPRC_low: epoch 2/30 val_F1=0.4273 val_AUC=0.8380 epoch_s=12.3 best=2
[2026-11-03T07:20:59] C17/X_LHPRC_low: epoch 3/30 val_F1=0.4326 val_AUC=0.8663 epoch_s=12.4 best=3
[2026-11-03T07:21:11] C17/X_LHPRC_low: epoch 4/30 val_F1=0.5214 val_AUC=0.9062 epoch_s=12.6 best=4
[2026-11-03T07:21:24] C17/X_LHPRC_low: epoch 5/30 val_F1=0.5224 val_AUC=0.8748 epoch_s=12.4 best=4
[2026-11-03T07:21:36] C17/X_LHPRC_low: epoch 6/30 val_F1=0.5929 val_AUC=0.8287 epoch_s=12.5 best=4
[2026-11-03T07:21:49] C17/X_LHPRC_low: epoch 7/30 val_F1=0.6402 val_AUC=0.8762 epoch_s=12.4 best=7
[2026-11-03T07:22:01] C17/X_LHPRC_low: epoch 8/30 val_F1=0.5556 val_AUC=0.8681 epoch_s=12.5 best=7
[2026-11-03T07:22:14] C17/X_LHPRC_low: epoch 9/30 val_F1=0.6527 val_AUC=0.9077 epoch_s=12.4 best=9
[2026-11-03T07:22:2

In [79]:
# C17 / X_LHPRC_before — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_before'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:26:37] C17/X_LHPRC_before: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:26:49] C17/X_LHPRC_before: epoch 1/30 val_F1=0.2000 val_AUC=0.6188 epoch_s=12.1 best=1
[2026-11-03T07:27:01] C17/X_LHPRC_before: epoch 2/30 val_F1=0.1739 val_AUC=0.6880 epoch_s=12.7 best=2
[2026-11-03T07:27:14] C17/X_LHPRC_before: epoch 3/30 val_F1=0.3992 val_AUC=0.7869 epoch_s=12.4 best=3
[2026-11-03T07:27:26] C17/X_LHPRC_before: epoch 4/30 val_F1=0.5150 val_AUC=0.8304 epoch_s=12.3 best=4
[2026-11-03T07:27:39] C17/X_LHPRC_before: epoch 5/30 val_F1=0.4371 val_AUC=0.8056 epoch_s=12.4 best=4
[2026-11-03T07:27:51] C17/X_LHPRC_before: epoch 6/30 val_F1=0.3969 val_AUC=0.7701 epoch_s=12.5 best=4
[2026-11-03T07:28:04] C17/X_LHPRC_before: epoch 7/30 val_F1=0.4198 val_AUC=0.8160 epoch_s=12.4 best=4
[2026-11-03T07:28:16] C17/X_LHPRC_before: epoch 8/30 val_F1=0.5645 val_AUC=0.8120 epoch_s=12.4 best=8
[2026-11-03T07:28:28] C17/X_LHPRC_before: epoch 9/30 val_F1=0.5426 val_AUC=0.8787 epoch_s=1

In [80]:
# C17 / X_LHPRS_mass90 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass90'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [81]:
# C17 / X_LHPR_lam001 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam001'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [82]:
# C17 / X_LHPR_lam010 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam010'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [83]:
# C17 / X_LHPR_seed43 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed43'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [84]:
# C17 / X_LHPR_seed44 — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed44'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [85]:
# C17 / X_L — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_L'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [86]:
# C17 / X_LH — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LH'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [87]:
# C17 / X_LP — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [88]:
# C17 / X_LR — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [89]:
# C17 / X_LS — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [90]:
# C17 / X_LC — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [91]:
# C17 / X_HS — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [92]:
# C17 / X_HC — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [93]:
# C17 / X_PS — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [94]:
# C17 / X_PC — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [95]:
# C17 / X_RS — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [96]:
# C17 / X_RC — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [97]:
# C17 / X_SC — only runs if this stage is selected in CELL 1.
if DATASET == 'C17' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_SC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [98]:
# C17 — validation comparison, saved after all selected stages.
if DATASET == 'C17' and DATASET in DATASETS:
    folder=RESULTS_ROOT/f'results_{DATASET}_ablation_comparison'
    folder.mkdir(parents=True,exist_ok=True)
    rows=[]
    for spec in STAGE_SPECS:
        path=stage_dir(spec.name)/'COMPLETED.json'
        if not path.exists():continue
        info=json.loads(path.read_text(encoding='utf-8'))
        if (info.get('spec')!=to_python(asdict(spec)) or
            info.get('n_epochs')!=ABLATION_EPOCHS):continue
        v=info['validation']
        rows.append({'stage':spec.name,'switches':'+'.join(spec.enabled),
           'val_accuracy':v['accuracy'],'val_macro_f1':v['macro_f1'],
           'val_macro_ovr_auc':v['macro_ovr_auc'],
           'best_epoch':info['best_epoch_1based'],
           'minutes':info['wall_seconds']/60})
    if rows:
        df=pd.DataFrame(rows).sort_values('val_macro_f1',ascending=False)
        df.to_csv(folder/'all_validation_results.csv',index=False)
        fig,ax=plt.subplots(figsize=(max(8,len(df)*.42),4))
        ax.bar(df.stage,df.val_macro_f1);ax.tick_params(axis='x',rotation=75)
        ax.set(title=f'C17: new ablation validation macro-F1',ylabel='Macro-F1')
        fig.tight_layout();fig.savefig(folder/'validation_macro_f1.png',dpi=140);plt.close(fig)
        print(df.to_string(index=False))
        progress('C17: comparisons saved to '+str(folder))
    else:
        progress('C17: no selected stages completed')
    del train_loader,val_loader,test_loader,train_data,val_data,test_data
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

              stage  switches  val_accuracy  val_macro_f1  val_macro_ovr_auc  best_epoch  minutes
                X_C         C     96.666667      0.910569           0.923750          12 5.760346
      X_LHPRS_top80 L+H+P+R+S     93.333333      0.833188           0.895055          10 6.014708
X_LHPRS_gatedmass80 L+H+P+R+S     93.333333      0.833188           0.897115          10 6.027648
 X_LHPRS_poolmass80 L+H+P+R+S     90.000000      0.813889           0.936799          25 6.095957
     X_LHPRS_mass80 L+H+P+R+S     90.000000      0.801786           0.931085          29 6.001571
               X_HP       H+P     90.000000      0.800000           0.911195           9 5.769136
               X_HR       H+R     90.000000      0.800000           0.919409          17 5.687397
      X_LHPRS_top20 L+H+P+R+S     90.000000      0.800000           0.943764          28 6.024155
             X_LHPR   L+H+P+R     90.000000      0.793651           0.916319          25 5.942768
                X_P 

---
## Dataset BRACS
This cell loads only BRACS's original course split/configuration. After initialization, **each experiment is an independent notebook cell**, so executed notebook results are saved between models and progress is written every epoch to `all_datasets_progress.log`.

In [99]:
# DATASET START: BRACS — original course config, loader, and train/validation/test splits.
DATASET = 'BRACS'
CONFIG_PATH=PROJECT_DIR/'config'/CONFIG_FILES[DATASET]
if DATASET in DATASETS:
    conf=load_config(str(CONFIG_PATH))
    seed_everything(int(conf.seed))
    if not Path(conf.data_dir).exists():
        raise FileNotFoundError('HDF5 feature file missing for '+DATASET+': '+str(conf.data_dir))
    train_data,val_data,test_data=build_HDF5_feat_dataset(conf.data_dir,conf)
    conf.D_feat=infer_feature_dim(train_data)
    if int(conf.B)!=1:
        raise ValueError('Original ABMIL ignores padding masks: use the starter batch size B=1')
    batch_size=conf.B;num_workers=int(conf.n_worker)
    pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
    train_loader=DataLoader(train_data,batch_size=batch_size,shuffle=True,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    val_loader=DataLoader(val_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    test_loader=DataLoader(test_data,batch_size=batch_size,shuffle=False,
        num_workers=num_workers,pin_memory=pin_memory,collate_fn=mil_collate_fn)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    CLASS_IDS=list(range(int(conf.n_class)))
    TRAIN_COUNTS=train_label_counts()
    NUM_EPOCHS=int(conf.train_epoch if EPOCH_OVERRIDE is None else EPOCH_OVERRIDE)
    ABLATION_EPOCHS=NUM_EPOCHS
    progress(f'BRACS: initialized train={len(train_data)} val={len(val_data)} test={len(test_data)} epochs={ABLATION_EPOCHS}')
    progress(f'BRACS: train class counts: {TRAIN_COUNTS.tolist()}')
else:
    progress('BRACS: skipped (not in DATASETS)')

[2026-11-03T07:32:53] BRACS: initialized train=389 val=62 test=86 epochs=30
[2026-11-03T07:32:53] BRACS: train class counts: [24, 119, 54, 24, 28, 40, 100]


In [100]:
# BRACS / A0 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A0'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [101]:
# BRACS / A1 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A1'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [102]:
# BRACS / A2 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A2'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [103]:
# BRACS / A3 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A3'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [104]:
# BRACS / A4 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'A4'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [105]:
# BRACS / X_ — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [106]:
# BRACS / X_H — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_H'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:32:54] BRACS/X_H: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:33:15] BRACS/X_H: epoch 1/30 val_F1=0.1522 val_AUC=0.6345 epoch_s=21.7 best=1
[2026-11-03T07:33:34] BRACS/X_H: epoch 2/30 val_F1=0.1579 val_AUC=0.6875 epoch_s=18.7 best=2
[2026-11-03T07:33:51] BRACS/X_H: epoch 3/30 val_F1=0.3136 val_AUC=0.7425 epoch_s=16.6 best=3
[2026-11-03T07:34:07] BRACS/X_H: epoch 4/30 val_F1=0.2814 val_AUC=0.7632 epoch_s=16.5 best=3
[2026-11-03T07:34:24] BRACS/X_H: epoch 5/30 val_F1=0.3271 val_AUC=0.7985 epoch_s=16.8 best=5
[2026-11-03T07:34:41] BRACS/X_H: epoch 6/30 val_F1=0.3917 val_AUC=0.8152 epoch_s=17.0 best=6
[2026-11-03T07:34:58] BRACS/X_H: epoch 7/30 val_F1=0.4311 val_AUC=0.8344 epoch_s=17.3 best=7
[2026-11-03T07:35:16] BRACS/X_H: epoch 8/30 val_F1=0.3926 val_AUC=0.8023 epoch_s=17.6 best=7
[2026-11-03T07:35:33] BRACS/X_H: epoch 9/30 val_F1=0.3980 val_AUC=0.8236 epoch_s=17.2 best=7
[2026-11-03T07:35:49] BRACS/X_H: epoch 10/30 val_F1=0.3775 val_AUC=0.8140 epoch

In [107]:
# BRACS / X_P — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_P'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:41:26] BRACS/X_P: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:41:42] BRACS/X_P: epoch 1/30 val_F1=0.1698 val_AUC=0.6891 epoch_s=16.6 best=1
[2026-11-03T07:42:00] BRACS/X_P: epoch 2/30 val_F1=0.3471 val_AUC=0.7635 epoch_s=17.1 best=2
[2026-11-03T07:42:16] BRACS/X_P: epoch 3/30 val_F1=0.3437 val_AUC=0.8042 epoch_s=16.9 best=3
[2026-11-03T07:42:33] BRACS/X_P: epoch 4/30 val_F1=0.3480 val_AUC=0.8128 epoch_s=16.8 best=4
[2026-11-03T07:42:51] BRACS/X_P: epoch 5/30 val_F1=0.3352 val_AUC=0.8274 epoch_s=17.2 best=5
[2026-11-03T07:43:08] BRACS/X_P: epoch 6/30 val_F1=0.4137 val_AUC=0.8353 epoch_s=17.3 best=6
[2026-11-03T07:43:25] BRACS/X_P: epoch 7/30 val_F1=0.4142 val_AUC=0.8292 epoch_s=17.0 best=6
[2026-11-03T07:43:42] BRACS/X_P: epoch 8/30 val_F1=0.4063 val_AUC=0.8220 epoch_s=16.9 best=6
[2026-11-03T07:43:59] BRACS/X_P: epoch 9/30 val_F1=0.3614 val_AUC=0.8252 epoch_s=17.1 best=6
[2026-11-03T07:44:16] BRACS/X_P: epoch 10/30 val_F1=0.3753 val_AUC=0.8126 epoch

In [108]:
# BRACS / X_R — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_R'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:50:04] BRACS/X_R: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:50:22] BRACS/X_R: epoch 1/30 val_F1=0.1698 val_AUC=0.6857 epoch_s=17.2 best=1
[2026-11-03T07:50:39] BRACS/X_R: epoch 2/30 val_F1=0.3064 val_AUC=0.7625 epoch_s=17.4 best=2
[2026-11-03T07:50:56] BRACS/X_R: epoch 3/30 val_F1=0.3437 val_AUC=0.8005 epoch_s=16.8 best=3
[2026-11-03T07:51:12] BRACS/X_R: epoch 4/30 val_F1=0.3480 val_AUC=0.8111 epoch_s=16.6 best=4
[2026-11-03T07:51:29] BRACS/X_R: epoch 5/30 val_F1=0.3406 val_AUC=0.8245 epoch_s=16.6 best=5
[2026-11-03T07:51:46] BRACS/X_R: epoch 6/30 val_F1=0.4115 val_AUC=0.8335 epoch_s=16.9 best=6
[2026-11-03T07:52:03] BRACS/X_R: epoch 7/30 val_F1=0.4143 val_AUC=0.8293 epoch_s=17.0 best=6
[2026-11-03T07:52:20] BRACS/X_R: epoch 8/30 val_F1=0.4063 val_AUC=0.8214 epoch_s=16.9 best=6
[2026-11-03T07:52:37] BRACS/X_R: epoch 9/30 val_F1=0.3551 val_AUC=0.8212 epoch_s=17.0 best=6
[2026-11-03T07:52:54] BRACS/X_R: epoch 10/30 val_F1=0.3593 val_AUC=0.8145 epoch

In [109]:
# BRACS / X_S — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_S'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T07:58:32] BRACS/X_S: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T07:58:48] BRACS/X_S: epoch 1/30 val_F1=0.1908 val_AUC=0.6794 epoch_s=16.2 best=1
[2026-11-03T07:59:05] BRACS/X_S: epoch 2/30 val_F1=0.1525 val_AUC=0.7392 epoch_s=16.3 best=2
[2026-11-03T07:59:21] BRACS/X_S: epoch 3/30 val_F1=0.3961 val_AUC=0.8085 epoch_s=16.3 best=3
[2026-11-03T07:59:38] BRACS/X_S: epoch 4/30 val_F1=0.3590 val_AUC=0.8213 epoch_s=16.7 best=3
[2026-11-03T07:59:54] BRACS/X_S: epoch 5/30 val_F1=0.3607 val_AUC=0.8262 epoch_s=16.7 best=3
[2026-11-03T08:00:11] BRACS/X_S: epoch 6/30 val_F1=0.3145 val_AUC=0.8228 epoch_s=16.8 best=3
[2026-11-03T08:00:28] BRACS/X_S: epoch 7/30 val_F1=0.3536 val_AUC=0.8227 epoch_s=17.0 best=3
[2026-11-03T08:00:45] BRACS/X_S: epoch 8/30 val_F1=0.3496 val_AUC=0.8122 epoch_s=17.0 best=3
[2026-11-03T08:01:02] BRACS/X_S: epoch 9/30 val_F1=0.3456 val_AUC=0.8105 epoch_s=16.7 best=3
[2026-11-03T08:01:19] BRACS/X_S: epoch 10/30 val_F1=0.3590 val_AUC=0.8254 epoch

In [110]:
# BRACS / X_C — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_C'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:06:53] BRACS/X_C: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:07:11] BRACS/X_C: epoch 1/30 val_F1=0.1630 val_AUC=0.7345 epoch_s=17.4 best=1
[2026-11-03T08:07:28] BRACS/X_C: epoch 2/30 val_F1=0.3010 val_AUC=0.7985 epoch_s=17.3 best=2
[2026-11-03T08:07:45] BRACS/X_C: epoch 3/30 val_F1=0.2930 val_AUC=0.8055 epoch_s=17.2 best=2
[2026-11-03T08:08:03] BRACS/X_C: epoch 4/30 val_F1=0.3557 val_AUC=0.8143 epoch_s=17.4 best=4
[2026-11-03T08:08:20] BRACS/X_C: epoch 5/30 val_F1=0.3883 val_AUC=0.8015 epoch_s=17.0 best=5
[2026-11-03T08:08:37] BRACS/X_C: epoch 6/30 val_F1=0.3718 val_AUC=0.8012 epoch_s=17.4 best=5
[2026-11-03T08:08:54] BRACS/X_C: epoch 7/30 val_F1=0.3714 val_AUC=0.8243 epoch_s=17.4 best=7
[2026-11-03T08:09:11] BRACS/X_C: epoch 8/30 val_F1=0.3609 val_AUC=0.8032 epoch_s=16.7 best=7
[2026-11-03T08:09:28] BRACS/X_C: epoch 9/30 val_F1=0.3876 val_AUC=0.8033 epoch_s=16.7 best=7
[2026-11-03T08:09:45] BRACS/X_C: epoch 10/30 val_F1=0.3716 val_AUC=0.8191 epoch

In [111]:
# BRACS / X_HP — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:15:31] BRACS/X_HP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:15:47] BRACS/X_HP: epoch 1/30 val_F1=0.1536 val_AUC=0.6614 epoch_s=16.5 best=1
[2026-11-03T08:16:05] BRACS/X_HP: epoch 2/30 val_F1=0.2484 val_AUC=0.7026 epoch_s=17.4 best=2
[2026-11-03T08:16:22] BRACS/X_HP: epoch 3/30 val_F1=0.2825 val_AUC=0.7646 epoch_s=17.3 best=3
[2026-11-03T08:16:40] BRACS/X_HP: epoch 4/30 val_F1=0.3908 val_AUC=0.8003 epoch_s=17.5 best=4
[2026-11-03T08:16:57] BRACS/X_HP: epoch 5/30 val_F1=0.4519 val_AUC=0.8116 epoch_s=17.7 best=5
[2026-11-03T08:17:15] BRACS/X_HP: epoch 6/30 val_F1=0.4151 val_AUC=0.8113 epoch_s=17.4 best=5
[2026-11-03T08:17:32] BRACS/X_HP: epoch 7/30 val_F1=0.4179 val_AUC=0.8203 epoch_s=17.6 best=5
[2026-11-03T08:17:49] BRACS/X_HP: epoch 8/30 val_F1=0.4242 val_AUC=0.8202 epoch_s=17.1 best=5
[2026-11-03T08:18:06] BRACS/X_HP: epoch 9/30 val_F1=0.4103 val_AUC=0.8124 epoch_s=17.0 best=5
[2026-11-03T08:18:24] BRACS/X_HP: epoch 10/30 val_F1=0.3563 val_AUC=0

In [112]:
# BRACS / X_HR — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:24:14] BRACS/X_HR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:24:31] BRACS/X_HR: epoch 1/30 val_F1=0.1536 val_AUC=0.6618 epoch_s=17.0 best=1
[2026-11-03T08:24:48] BRACS/X_HR: epoch 2/30 val_F1=0.2433 val_AUC=0.7001 epoch_s=17.0 best=2
[2026-11-03T08:25:05] BRACS/X_HR: epoch 3/30 val_F1=0.2823 val_AUC=0.7618 epoch_s=17.1 best=3
[2026-11-03T08:25:22] BRACS/X_HR: epoch 4/30 val_F1=0.4146 val_AUC=0.8000 epoch_s=17.1 best=4
[2026-11-03T08:25:39] BRACS/X_HR: epoch 5/30 val_F1=0.4513 val_AUC=0.8134 epoch_s=17.0 best=5
[2026-11-03T08:25:56] BRACS/X_HR: epoch 6/30 val_F1=0.4151 val_AUC=0.8124 epoch_s=17.1 best=5
[2026-11-03T08:26:13] BRACS/X_HR: epoch 7/30 val_F1=0.4179 val_AUC=0.8199 epoch_s=16.9 best=5
[2026-11-03T08:26:30] BRACS/X_HR: epoch 8/30 val_F1=0.4242 val_AUC=0.8223 epoch_s=16.7 best=5
[2026-11-03T08:26:46] BRACS/X_HR: epoch 9/30 val_F1=0.4138 val_AUC=0.8138 epoch_s=16.8 best=5
[2026-11-03T08:27:03] BRACS/X_HR: epoch 10/30 val_F1=0.3567 val_AUC=0

In [113]:
# BRACS / X_PR — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:32:44] BRACS/X_PR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:33:02] BRACS/X_PR: epoch 1/30 val_F1=0.1924 val_AUC=0.6875 epoch_s=17.5 best=1
[2026-11-03T08:33:20] BRACS/X_PR: epoch 2/30 val_F1=0.3064 val_AUC=0.7631 epoch_s=17.5 best=2
[2026-11-03T08:33:37] BRACS/X_PR: epoch 3/30 val_F1=0.3455 val_AUC=0.7990 epoch_s=17.6 best=3
[2026-11-03T08:33:55] BRACS/X_PR: epoch 4/30 val_F1=0.3461 val_AUC=0.8109 epoch_s=17.5 best=4
[2026-11-03T08:34:12] BRACS/X_PR: epoch 5/30 val_F1=0.3393 val_AUC=0.8256 epoch_s=17.6 best=5
[2026-11-03T08:34:30] BRACS/X_PR: epoch 6/30 val_F1=0.4106 val_AUC=0.8319 epoch_s=17.7 best=6
[2026-11-03T08:34:47] BRACS/X_PR: epoch 7/30 val_F1=0.4143 val_AUC=0.8275 epoch_s=17.5 best=6
[2026-11-03T08:35:05] BRACS/X_PR: epoch 8/30 val_F1=0.4086 val_AUC=0.8204 epoch_s=17.6 best=6
[2026-11-03T08:35:23] BRACS/X_PR: epoch 9/30 val_F1=0.3777 val_AUC=0.8214 epoch_s=17.7 best=6
[2026-11-03T08:35:41] BRACS/X_PR: epoch 10/30 val_F1=0.3728 val_AUC=0

In [114]:
# BRACS / X_LHP — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:41:41] BRACS/X_LHP: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:41:58] BRACS/X_LHP: epoch 1/30 val_F1=0.1811 val_AUC=0.6576 epoch_s=16.9 best=1
[2026-11-03T08:42:15] BRACS/X_LHP: epoch 2/30 val_F1=0.1579 val_AUC=0.6980 epoch_s=17.1 best=2
[2026-11-03T08:42:32] BRACS/X_LHP: epoch 3/30 val_F1=0.3576 val_AUC=0.7746 epoch_s=17.4 best=3
[2026-11-03T08:42:49] BRACS/X_LHP: epoch 4/30 val_F1=0.3540 val_AUC=0.8052 epoch_s=17.0 best=4
[2026-11-03T08:43:06] BRACS/X_LHP: epoch 5/30 val_F1=0.4262 val_AUC=0.8146 epoch_s=17.0 best=5
[2026-11-03T08:43:23] BRACS/X_LHP: epoch 6/30 val_F1=0.4713 val_AUC=0.8203 epoch_s=17.0 best=6
[2026-11-03T08:43:40] BRACS/X_LHP: epoch 7/30 val_F1=0.3866 val_AUC=0.8207 epoch_s=17.1 best=6
[2026-11-03T08:43:57] BRACS/X_LHP: epoch 8/30 val_F1=0.3874 val_AUC=0.8234 epoch_s=16.9 best=6
[2026-11-03T08:44:14] BRACS/X_LHP: epoch 9/30 val_F1=0.3793 val_AUC=0.8143 epoch_s=17.0 best=6
[2026-11-03T08:44:32] BRACS/X_LHP: epoch 10/30 val_F1=0.307

In [115]:
# BRACS / X_LHPR — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:50:21] BRACS/X_LHPR: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:50:38] BRACS/X_LHPR: epoch 1/30 val_F1=0.1819 val_AUC=0.6615 epoch_s=17.0 best=1
[2026-11-03T08:50:55] BRACS/X_LHPR: epoch 2/30 val_F1=0.1579 val_AUC=0.7004 epoch_s=17.3 best=2
[2026-11-03T08:51:13] BRACS/X_LHPR: epoch 3/30 val_F1=0.3569 val_AUC=0.7722 epoch_s=17.6 best=3
[2026-11-03T08:51:31] BRACS/X_LHPR: epoch 4/30 val_F1=0.3630 val_AUC=0.8041 epoch_s=17.5 best=4
[2026-11-03T08:51:48] BRACS/X_LHPR: epoch 5/30 val_F1=0.4259 val_AUC=0.8180 epoch_s=17.3 best=5
[2026-11-03T08:52:05] BRACS/X_LHPR: epoch 6/30 val_F1=0.4698 val_AUC=0.8228 epoch_s=17.4 best=6
[2026-11-03T08:52:23] BRACS/X_LHPR: epoch 7/30 val_F1=0.3883 val_AUC=0.8245 epoch_s=17.4 best=6
[2026-11-03T08:52:40] BRACS/X_LHPR: epoch 8/30 val_F1=0.4012 val_AUC=0.8274 epoch_s=17.4 best=6
[2026-11-03T08:52:58] BRACS/X_LHPR: epoch 9/30 val_F1=0.3707 val_AUC=0.8227 epoch_s=17.5 best=6
[2026-11-03T08:53:15] BRACS/X_LHPR: epoch 10/30 v

In [116]:
# BRACS / X_LHPRS_mass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T08:59:09] BRACS/X_LHPRS_mass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T08:59:26] BRACS/X_LHPRS_mass80: epoch 1/30 val_F1=0.1818 val_AUC=0.6601 epoch_s=17.7 best=1
[2026-11-03T08:59:44] BRACS/X_LHPRS_mass80: epoch 2/30 val_F1=0.1579 val_AUC=0.7000 epoch_s=17.8 best=2
[2026-11-03T09:00:02] BRACS/X_LHPRS_mass80: epoch 3/30 val_F1=0.3596 val_AUC=0.7707 epoch_s=18.1 best=3
[2026-11-03T09:00:21] BRACS/X_LHPRS_mass80: epoch 4/30 val_F1=0.3540 val_AUC=0.8049 epoch_s=18.5 best=4
[2026-11-03T09:00:41] BRACS/X_LHPRS_mass80: epoch 5/30 val_F1=0.4259 val_AUC=0.8171 epoch_s=19.6 best=5
[2026-11-03T09:00:59] BRACS/X_LHPRS_mass80: epoch 6/30 val_F1=0.4698 val_AUC=0.8220 epoch_s=18.1 best=6
[2026-11-03T09:01:17] BRACS/X_LHPRS_mass80: epoch 7/30 val_F1=0.3715 val_AUC=0.8232 epoch_s=18.1 best=6
[2026-11-03T09:01:35] BRACS/X_LHPRS_mass80: epoch 8/30 val_F1=0.4035 val_AUC=0.8277 epoch_s=18.2 best=6
[2026-11-03T09:01:53] BRACS/X_LHPRS_mass80: epoch 9/30 val_F1=0.3707 val_

In [117]:
# BRACS / X_LHPRS_top20 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top20'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:08:22] BRACS/X_LHPRS_top20: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:08:39] BRACS/X_LHPRS_top20: epoch 1/30 val_F1=0.1851 val_AUC=0.6582 epoch_s=17.5 best=1
[2026-11-03T09:08:57] BRACS/X_LHPRS_top20: epoch 2/30 val_F1=0.1579 val_AUC=0.6987 epoch_s=18.0 best=2
[2026-11-03T09:09:15] BRACS/X_LHPRS_top20: epoch 3/30 val_F1=0.3701 val_AUC=0.7690 epoch_s=18.2 best=3
[2026-11-03T09:09:33] BRACS/X_LHPRS_top20: epoch 4/30 val_F1=0.3657 val_AUC=0.8044 epoch_s=17.7 best=4
[2026-11-03T09:09:51] BRACS/X_LHPRS_top20: epoch 5/30 val_F1=0.4259 val_AUC=0.8161 epoch_s=17.7 best=5
[2026-11-03T09:10:09] BRACS/X_LHPRS_top20: epoch 6/30 val_F1=0.4713 val_AUC=0.8213 epoch_s=17.8 best=6
[2026-11-03T09:10:27] BRACS/X_LHPRS_top20: epoch 7/30 val_F1=0.3728 val_AUC=0.8200 epoch_s=18.2 best=6
[2026-11-03T09:10:45] BRACS/X_LHPRS_top20: epoch 8/30 val_F1=0.3897 val_AUC=0.8266 epoch_s=18.0 best=6
[2026-11-03T09:11:03] BRACS/X_LHPRS_top20: epoch 9/30 val_F1=0.3987 val_AUC=0.8181

In [118]:
# BRACS / X_LHPRS_top80 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:17:23] BRACS/X_LHPRS_top80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:17:41] BRACS/X_LHPRS_top80: epoch 1/30 val_F1=0.1819 val_AUC=0.6608 epoch_s=18.0 best=1
[2026-11-03T09:18:00] BRACS/X_LHPRS_top80: epoch 2/30 val_F1=0.1579 val_AUC=0.7000 epoch_s=18.5 best=2
[2026-11-03T09:18:18] BRACS/X_LHPRS_top80: epoch 3/30 val_F1=0.3596 val_AUC=0.7707 epoch_s=18.1 best=3
[2026-11-03T09:18:35] BRACS/X_LHPRS_top80: epoch 4/30 val_F1=0.3540 val_AUC=0.8042 epoch_s=17.6 best=4
[2026-11-03T09:18:53] BRACS/X_LHPRS_top80: epoch 5/30 val_F1=0.4259 val_AUC=0.8180 epoch_s=17.8 best=5
[2026-11-03T09:19:11] BRACS/X_LHPRS_top80: epoch 6/30 val_F1=0.4698 val_AUC=0.8218 epoch_s=18.0 best=6
[2026-11-03T09:19:29] BRACS/X_LHPRS_top80: epoch 7/30 val_F1=0.3866 val_AUC=0.8242 epoch_s=17.9 best=6
[2026-11-03T09:19:47] BRACS/X_LHPRS_top80: epoch 8/30 val_F1=0.4012 val_AUC=0.8282 epoch_s=18.2 best=6
[2026-11-03T09:20:06] BRACS/X_LHPRS_top80: epoch 9/30 val_F1=0.3707 val_AUC=0.8211

In [119]:
# BRACS / X_LHPRS_top16 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_top16'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:26:33] BRACS/X_LHPRS_top16: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:26:50] BRACS/X_LHPRS_top16: epoch 1/30 val_F1=0.1853 val_AUC=0.6574 epoch_s=17.3 best=1
[2026-11-03T09:27:08] BRACS/X_LHPRS_top16: epoch 2/30 val_F1=0.1765 val_AUC=0.7010 epoch_s=18.1 best=2
[2026-11-03T09:27:26] BRACS/X_LHPRS_top16: epoch 3/30 val_F1=0.3952 val_AUC=0.7712 epoch_s=17.8 best=3
[2026-11-03T09:27:44] BRACS/X_LHPRS_top16: epoch 4/30 val_F1=0.3696 val_AUC=0.8027 epoch_s=17.6 best=4
[2026-11-03T09:28:02] BRACS/X_LHPRS_top16: epoch 5/30 val_F1=0.4262 val_AUC=0.8176 epoch_s=18.0 best=5
[2026-11-03T09:28:20] BRACS/X_LHPRS_top16: epoch 6/30 val_F1=0.4549 val_AUC=0.8191 epoch_s=17.9 best=6
[2026-11-03T09:28:38] BRACS/X_LHPRS_top16: epoch 7/30 val_F1=0.3747 val_AUC=0.8192 epoch_s=18.1 best=6
[2026-11-03T09:28:56] BRACS/X_LHPRS_top16: epoch 8/30 val_F1=0.3897 val_AUC=0.8269 epoch_s=18.1 best=6
[2026-11-03T09:29:14] BRACS/X_LHPRS_top16: epoch 9/30 val_F1=0.3945 val_AUC=0.8160

In [120]:
# BRACS / X_LHPRS_gatedmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_gatedmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:35:39] BRACS/X_LHPRS_gatedmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:35:57] BRACS/X_LHPRS_gatedmass80: epoch 1/30 val_F1=0.1818 val_AUC=0.6608 epoch_s=18.0 best=1
[2026-11-03T09:36:15] BRACS/X_LHPRS_gatedmass80: epoch 2/30 val_F1=0.1579 val_AUC=0.7003 epoch_s=18.1 best=2
[2026-11-03T09:36:33] BRACS/X_LHPRS_gatedmass80: epoch 3/30 val_F1=0.3701 val_AUC=0.7710 epoch_s=18.0 best=3
[2026-11-03T09:36:51] BRACS/X_LHPRS_gatedmass80: epoch 4/30 val_F1=0.3540 val_AUC=0.8051 epoch_s=17.8 best=4
[2026-11-03T09:37:09] BRACS/X_LHPRS_gatedmass80: epoch 5/30 val_F1=0.4259 val_AUC=0.8165 epoch_s=18.2 best=5
[2026-11-03T09:37:27] BRACS/X_LHPRS_gatedmass80: epoch 6/30 val_F1=0.4698 val_AUC=0.8220 epoch_s=18.3 best=6
[2026-11-03T09:37:45] BRACS/X_LHPRS_gatedmass80: epoch 7/30 val_F1=0.3715 val_AUC=0.8232 epoch_s=17.6 best=6
[2026-11-03T09:38:03] BRACS/X_LHPRS_gatedmass80: epoch 8/30 val_F1=0.4035 val_AUC=0.8283 epoch_s=17.8 best=6
[2026-11-03T09:38:21] BRACS/

In [121]:
# BRACS / X_LHPRS_poolmass80 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_poolmass80'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:44:41] BRACS/X_LHPRS_poolmass80: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:44:59] BRACS/X_LHPRS_poolmass80: epoch 1/30 val_F1=0.2292 val_AUC=0.6705 epoch_s=17.8 best=1
[2026-11-03T09:45:17] BRACS/X_LHPRS_poolmass80: epoch 2/30 val_F1=0.1791 val_AUC=0.7071 epoch_s=18.3 best=1
[2026-11-03T09:45:36] BRACS/X_LHPRS_poolmass80: epoch 3/30 val_F1=0.3521 val_AUC=0.7746 epoch_s=18.4 best=3
[2026-11-03T09:45:54] BRACS/X_LHPRS_poolmass80: epoch 4/30 val_F1=0.3546 val_AUC=0.8062 epoch_s=18.2 best=4
[2026-11-03T09:46:12] BRACS/X_LHPRS_poolmass80: epoch 5/30 val_F1=0.4417 val_AUC=0.8153 epoch_s=18.0 best=5
[2026-11-03T09:46:30] BRACS/X_LHPRS_poolmass80: epoch 6/30 val_F1=0.4360 val_AUC=0.8225 epoch_s=17.8 best=6
[2026-11-03T09:46:48] BRACS/X_LHPRS_poolmass80: epoch 7/30 val_F1=0.3715 val_AUC=0.8265 epoch_s=18.0 best=6
[2026-11-03T09:47:06] BRACS/X_LHPRS_poolmass80: epoch 8/30 val_F1=0.3859 val_AUC=0.8262 epoch_s=18.0 best=6
[2026-11-03T09:47:24] BRACS/X_LHPRS_p

In [122]:
# BRACS / X_LHPRC_low — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_low'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T09:53:50] BRACS/X_LHPRC_low: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T09:54:09] BRACS/X_LHPRC_low: epoch 1/30 val_F1=0.2766 val_AUC=0.6846 epoch_s=18.8 best=1
[2026-11-03T09:54:27] BRACS/X_LHPRC_low: epoch 2/30 val_F1=0.3168 val_AUC=0.7638 epoch_s=18.4 best=2
[2026-11-03T09:54:46] BRACS/X_LHPRC_low: epoch 3/30 val_F1=0.2643 val_AUC=0.7972 epoch_s=18.7 best=2
[2026-11-03T09:55:05] BRACS/X_LHPRC_low: epoch 4/30 val_F1=0.3799 val_AUC=0.8118 epoch_s=18.7 best=4
[2026-11-03T09:55:24] BRACS/X_LHPRC_low: epoch 5/30 val_F1=0.3434 val_AUC=0.8074 epoch_s=18.9 best=4
[2026-11-03T09:55:43] BRACS/X_LHPRC_low: epoch 6/30 val_F1=0.3877 val_AUC=0.7913 epoch_s=19.0 best=4
[2026-11-03T09:56:02] BRACS/X_LHPRC_low: epoch 7/30 val_F1=0.3194 val_AUC=0.8105 epoch_s=18.9 best=4
[2026-11-03T09:56:21] BRACS/X_LHPRC_low: epoch 8/30 val_F1=0.3933 val_AUC=0.8176 epoch_s=18.8 best=8
[2026-11-03T09:56:39] BRACS/X_LHPRC_low: epoch 9/30 val_F1=0.4174 val_AUC=0.8201 epoch_s=18.3 best=9

In [123]:
# BRACS / X_LHPRC_before — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRC_before'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

[2026-11-03T10:02:43] BRACS/X_LHPRC_before: START, epochs 1–30, patches as stored, device=cuda
[2026-11-03T10:02:56] BRACS/X_LHPRC_before: epoch 1/30 val_F1=0.2689 val_AUC=0.6614 epoch_s=13.7 best=1
[2026-11-03T10:03:10] BRACS/X_LHPRC_before: epoch 2/30 val_F1=0.2165 val_AUC=0.6940 epoch_s=13.7 best=1
[2026-11-03T10:03:23] BRACS/X_LHPRC_before: epoch 3/30 val_F1=0.3006 val_AUC=0.7578 epoch_s=13.5 best=3
[2026-11-03T10:03:37] BRACS/X_LHPRC_before: epoch 4/30 val_F1=0.4152 val_AUC=0.7973 epoch_s=13.6 best=4
[2026-11-03T10:03:51] BRACS/X_LHPRC_before: epoch 5/30 val_F1=0.3399 val_AUC=0.8073 epoch_s=13.7 best=4
[2026-11-03T10:04:04] BRACS/X_LHPRC_before: epoch 6/30 val_F1=0.4213 val_AUC=0.8011 epoch_s=13.3 best=6
[2026-11-03T10:04:18] BRACS/X_LHPRC_before: epoch 7/30 val_F1=0.3472 val_AUC=0.8189 epoch_s=13.4 best=6
[2026-11-03T10:04:31] BRACS/X_LHPRC_before: epoch 8/30 val_F1=0.3950 val_AUC=0.8118 epoch_s=13.6 best=6
[2026-11-03T10:04:45] BRACS/X_LHPRC_before: epoch 9/30 val_F1=0.3641 val_

In [124]:
# BRACS / X_LHPRS_mass90 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPRS_mass90'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [125]:
# BRACS / X_LHPR_lam001 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam001'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [126]:
# BRACS / X_LHPR_lam010 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_lam010'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [127]:
# BRACS / X_LHPR_seed43 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed43'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [128]:
# BRACS / X_LHPR_seed44 — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LHPR_seed44'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [129]:
# BRACS / X_L — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_L'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [130]:
# BRACS / X_LH — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LH'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [131]:
# BRACS / X_LP — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LP'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [132]:
# BRACS / X_LR — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LR'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [133]:
# BRACS / X_LS — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [134]:
# BRACS / X_LC — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_LC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [135]:
# BRACS / X_HS — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [136]:
# BRACS / X_HC — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_HC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [137]:
# BRACS / X_PS — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [138]:
# BRACS / X_PC — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_PC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [139]:
# BRACS / X_RS — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RS'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [140]:
# BRACS / X_RC — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_RC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [141]:
# BRACS / X_SC — only runs if this stage is selected in CELL 1.
if DATASET == 'BRACS' and DATASET in DATASETS:
    selected_spec = next((s for s in STAGE_SPECS if s.name == 'X_SC'), None)
    if selected_spec is not None:
        train_ablation(selected_spec)

In [142]:
# BRACS — validation comparison, saved after all selected stages.
if DATASET == 'BRACS' and DATASET in DATASETS:
    folder=RESULTS_ROOT/f'results_{DATASET}_ablation_comparison'
    folder.mkdir(parents=True,exist_ok=True)
    rows=[]
    for spec in STAGE_SPECS:
        path=stage_dir(spec.name)/'COMPLETED.json'
        if not path.exists():continue
        info=json.loads(path.read_text(encoding='utf-8'))
        if (info.get('spec')!=to_python(asdict(spec)) or
            info.get('n_epochs')!=ABLATION_EPOCHS):continue
        v=info['validation']
        rows.append({'stage':spec.name,'switches':'+'.join(spec.enabled),
           'val_accuracy':v['accuracy'],'val_macro_f1':v['macro_f1'],
           'val_macro_ovr_auc':v['macro_ovr_auc'],
           'best_epoch':info['best_epoch_1based'],
           'minutes':info['wall_seconds']/60})
    if rows:
        df=pd.DataFrame(rows).sort_values('val_macro_f1',ascending=False)
        df.to_csv(folder/'all_validation_results.csv',index=False)
        fig,ax=plt.subplots(figsize=(max(8,len(df)*.42),4))
        ax.bar(df.stage,df.val_macro_f1);ax.tick_params(axis='x',rotation=75)
        ax.set(title=f'BRACS: new ablation validation macro-F1',ylabel='Macro-F1')
        fig.tight_layout();fig.savefig(folder/'validation_macro_f1.png',dpi=140);plt.close(fig)
        print(df.to_string(index=False))
        progress('BRACS: comparisons saved to '+str(folder))
    else:
        progress('BRACS: no selected stages completed')
    del train_loader,val_loader,test_loader,train_data,val_data,test_data
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

              stage  switches  val_accuracy  val_macro_f1  val_macro_ovr_auc  best_epoch  minutes
      X_LHPRS_top20 L+H+P+R+S     53.225806      0.471336           0.821289           6 8.965773
              X_LHP     L+H+P     53.225806      0.471336           0.820323           6 8.608018
X_LHPRS_gatedmass80 L+H+P+R+S     53.225806      0.469778           0.821975           6 8.978080
      X_LHPRS_top80 L+H+P+R+S     53.225806      0.469778           0.821809           6 9.102539
     X_LHPRS_mass80 L+H+P+R+S     53.225806      0.469778           0.822047           6 9.155150
             X_LHPR   L+H+P+R     53.225806      0.469778           0.822770           6 8.735901
        X_LHPRC_low L+H+P+R+C     53.225806      0.466750           0.819025          10 8.821975
     X_LHPRC_before L+H+P+R+C     51.612903      0.464021           0.814976          14 6.794880
      X_LHPRS_top16 L+H+P+R+S     51.612903      0.454880           0.819115           6 9.042800
               X_HP 

In [143]:

# ============================================================
# NEW CELL: TEST ALL SAVED MODELS — NO TRAINING
# C16 -> C17 -> BRACS
# ============================================================

from pathlib import Path
import json
import gc
import traceback

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

TEST_DATASETS = ('C16', 'C17', 'BRACS')

# Avoid evaluating models twice.
SKIP_ALREADY_TESTED = True

# Accept a completed marker or the older final metrics file.
REQUIRE_COMPLETED = True


def test_saved_checkpoints(dataset_name):

    global DATASET, conf, device, CLASS_IDS

    DATASET = dataset_name

    # --------------------------------------------------------
    # 1. Recreate the original dataset configuration
    # --------------------------------------------------------
    conf = load_config(
        str(PROJECT_DIR / 'config' / CONFIG_FILES[DATASET])
    )

    train_ds, val_ds, test_ds = build_HDF5_feat_dataset(
        conf.data_dir, conf
    )

    conf.D_feat = infer_feature_dim(train_ds)

    device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

    CLASS_IDS = list(range(int(conf.n_class)))

    # Only the TEST DataLoader is needed.
    test_dl = DataLoader(
        test_ds,
        batch_size=int(conf.B),
        shuffle=False,
        num_workers=int(conf.n_worker),
        pin_memory=(
            bool(conf.pin_memory) and torch.cuda.is_available()
        ),
        collate_fn=mil_collate_fn
    )

    rows = []

    folders = sorted(
        RESULTS_ROOT.glob(f'results_{DATASET}_*')
    )

    print(
        f'\n[{DATASET}] Searching saved checkpoints...',
        flush=True
    )

    # --------------------------------------------------------
    # 2. Loop through existing trained models
    # --------------------------------------------------------
    for folder in folders:

        checkpoint_path = folder / 'best_model.pt'

        if not checkpoint_path.is_file():
            continue

        if REQUIRE_COMPLETED:
            completed = (folder / 'COMPLETED.json').exists()
            legacy_completed = (folder / 'metrics.json').exists()

            if not completed and not legacy_completed:
                print('SKIP unfinished:', folder.name)
                continue

        metrics_path = folder / 'test_metrics.json'
        predictions_path = folder / 'test_predictions.csv'

        if (
            SKIP_ALREADY_TESTED
            and metrics_path.exists()
            and predictions_path.exists()
        ):
            metrics = json.loads(
                metrics_path.read_text(encoding='utf-8')
            )
            rows.append({
                'stage': folder.name[len(f'results_{DATASET}_'):],
                **metrics
            })
            print('ALREADY TESTED:', folder.name)
            continue

        net = None

        try:
            # ------------------------------------------------
            # 3. Load validation-selected model weights
            # ------------------------------------------------
            checkpoint = torch.load(
                checkpoint_path,
                map_location='cpu',
                weights_only=True
            )

            if (
                not isinstance(checkpoint, dict)
                or 'model_state_dict' not in checkpoint
            ):
                print('SKIP incompatible:', folder.name)
                continue

            if checkpoint.get('dataset', DATASET) != DATASET:
                print('SKIP dataset mismatch:', folder.name)
                continue

            # New ablation checkpoints store the full spec.
            if 'spec' in checkpoint:
                spec = AblationSpec(**checkpoint['spec'])

            else:
                # Older A0-A4 checkpoints do not store "spec".
                # Reconstruct their known original configurations.
                spec = next(
                    (
                        s for s in ORIGINAL_STAGE_SPECS
                        if s.name == checkpoint.get('stage')
                    ),
                    None
                )

                if spec is None:
                    print('SKIP unknown legacy stage:', folder.name)
                    continue

            expected_folder = f'results_{DATASET}_{spec.name}'

            if folder.name != expected_folder:
                print('SKIP stage mismatch:', folder.name)
                continue

            # Reuse the already-defined architecture.
            net = AblationABMIL(conf, spec).to(device)

            net.load_state_dict(
                checkpoint['model_state_dict'],
                strict=True
            )

            net.eval()

            # ------------------------------------------------
            # 4. Perform TEST inference (no gradients)
            # ------------------------------------------------
            # This existing helper already uses torch.no_grad().
            # It also saves test_predictions.csv and
            # test_retention_statistics.csv.
            y_true, probabilities = collect_ablation_details(
                net,
                test_dl,
                folder,
                split='test'
            )

            y_pred = probabilities.argmax(axis=1)

            # ------------------------------------------------
            # 5. Compute test metrics
            # ------------------------------------------------
            metrics = prediction_metrics(
                y_true, probabilities
            )

            # Cross-entropy from unadjusted class probabilities.
            true_probabilities = probabilities[
                np.arange(len(y_true)), y_true
            ]

            metrics['test_loss_raw_ce'] = float(
                -np.log(
                    np.clip(true_probabilities, 1e-12, 1.0)
                ).mean()
            )

            if 'best_epoch' in checkpoint:
                metrics['best_epoch_1based'] = (
                    int(checkpoint['best_epoch']) + 1
                )
            else:
                metrics['best_epoch_1based'] = (
                    checkpoint.get('best_epoch_1based')
                )

            metrics['dataset'] = DATASET
            metrics['stage'] = spec.name

            metrics_path.write_text(
                json.dumps(to_python(metrics), indent=2),
                encoding='utf-8'
            )

            # Classification report
            report = classification_report(
                y_true,
                y_pred,
                labels=CLASS_IDS,
                zero_division=0
            )

            (
                folder / 'test_classification_report.txt'
            ).write_text(report, encoding='utf-8')

            # ------------------------------------------------
            # 6. Test confusion matrix
            # ------------------------------------------------
            cm = confusion_matrix(
                y_true, y_pred, labels=CLASS_IDS
            )

            pd.DataFrame(
                cm,
                index=CLASS_IDS,
                columns=CLASS_IDS
            ).to_csv(
                folder / 'test_confusion_matrix.csv'
            )

            fig, ax = plt.subplots(figsize=(6, 5))

            ConfusionMatrixDisplay(
                cm,
                display_labels=CLASS_IDS
            ).plot(
                ax=ax,
                cmap='Blues',
                colorbar=False
            )

            ax.set_title(
                f'{DATASET} / {spec.name} - TEST confusion matrix'
            )

            fig.tight_layout()

            fig.savefig(
                folder / 'test_confusion_matrix.png',
                dpi=160
            )

            plt.close(fig)

            # ------------------------------------------------
            # 7. Test ROC curves
            # ------------------------------------------------
            fig, ax = plt.subplots(figsize=(6, 4.5))

            for c in CLASS_IDS:

                binary = (y_true == c).astype(int)

                if len(np.unique(binary)) < 2:
                    continue

                fpr, tpr, _ = roc_curve(
                    binary, probabilities[:, c]
                )

                auc = roc_auc_score(
                    binary, probabilities[:, c]
                )

                ax.plot(
                    fpr,
                    tpr,
                    label=f'Class {c} (AUC={auc:.3f})'
                )

            ax.plot(
                [0, 1], [0, 1],
                '--', color='gray', lw=1
            )

            ax.set(
                xlabel='False positive rate',
                ylabel='True positive rate',
                title=f'{DATASET} / {spec.name} - TEST ROC'
            )

            ax.legend(fontsize=8)

            fig.tight_layout()

            fig.savefig(
                folder / 'test_roc.png',
                dpi=160
            )

            plt.close(fig)

            # ------------------------------------------------
            # 8. Test per-class recall
            # ------------------------------------------------
            recalls = [
                metrics['class_recall'][str(c)]
                for c in CLASS_IDS
            ]

            fig, ax = plt.subplots(figsize=(6, 4))

            ax.bar(
                [str(c) for c in CLASS_IDS],
                [
                    100 * r if r is not None else 0
                    for r in recalls
                ]
            )

            ax.set(
                ylim=(0, 100),
                xlabel='Class',
                ylabel='Recall (%)',
                title=f'{DATASET} / {spec.name} - TEST recall'
            )

            fig.tight_layout()

            fig.savefig(
                folder / 'test_per_class_recall.png',
                dpi=160
            )

            plt.close(fig)

            # ------------------------------------------------
            # 9. Store result for dataset comparison
            # ------------------------------------------------
            rows.append({
                'stage': spec.name,
                **metrics
            })

            print(
                f'DONE {DATASET}/{spec.name} | '
                f'Accuracy={metrics["accuracy"]:.2f}% | '
                f'Macro-F1={metrics["macro_f1"]:.4f} | '
                f'AUC={metrics["macro_ovr_auc"]}',
                flush=True
            )

        except Exception:

            print(
                f'ERROR testing {folder.name}; '
                f'see test_error.txt',
                flush=True
            )

            (folder / 'test_error.txt').write_text(
                traceback.format_exc(),
                encoding='utf-8'
            )

        finally:

            if net is not None:
                del net

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 10. Save the COMPLETE dataset test comparison
    # --------------------------------------------------------
    if rows:

        summary = pd.DataFrame(rows)

        out_dir = (
            RESULTS_ROOT
            / f'results_{DATASET}_ablation_comparison'
        )

        out_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        summary.sort_values(
            'macro_f1',
            ascending=False
        ).to_csv(
            out_dir / 'all_test_results.csv',
            index=False
        )

        print(
            f'[{DATASET}] Saved test comparison:',
            out_dir / 'all_test_results.csv'
        )

    else:
        print(
            f'[{DATASET}] No compatible completed models found.'
        )

    del test_dl, train_ds, val_ds, test_ds
    gc.collect()

    return rows


# ============================================================
# EXECUTE TEST-ONLY EVALUATION FOR ALL THREE DATASETS
# ============================================================

ALL_TEST_RESULTS = {}

for _dataset in TEST_DATASETS:

    ALL_TEST_RESULTS[_dataset] = test_saved_checkpoints(
        _dataset
    )

print(
    '\nTEST-ONLY EVALUATION FINISHED.'
    '\nNO MODELS WERE RETRAINED.'
)



[C16] Searching saved checkpoints...
DONE C16/X_C | Accuracy=90.70% | Macro-F1=0.8996 | AUC=0.9517857142857142
DONE C16/X_H | Accuracy=93.02% | Macro-F1=0.9236 | AUC=0.9683673469387755
DONE C16/X_HP | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.9505102040816327
DONE C16/X_HR | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.952295918367347
DONE C16/X_LHP | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.95
DONE C16/X_LHPR | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.9530612244897959
DONE C16/X_LHPRC_before | Accuracy=90.70% | Macro-F1=0.8996 | AUC=0.9326530612244899
DONE C16/X_LHPRC_low | Accuracy=90.70% | Macro-F1=0.8986 | AUC=0.9591836734693877
DONE C16/X_LHPRS_gatedmass80 | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.9530612244897959
DONE C16/X_LHPRS_mass80 | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.9525510204081633
DONE C16/X_LHPRS_poolmass80 | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.950765306122449
DONE C16/X_LHPRS_top16 | Accuracy=88.37% | Macro-F1=0.8668 | AUC=0.9502551020408163
DONE C16/X_L

## Finished

Open the saved `...EXECUTED.ipynb` after execution. Each stage's checkpoint, validation predictions, learning curves, confusion matrix, ROC and patch-retention histogram are saved under `results_{Dataset}_{Stage}/`. Summaries appear under `results_{Dataset}_ablation_comparison/`. Training prints each epoch to `all_datasets_progress.log`, which can be monitored while the runner is active.

Do **not** choose a new architecture based on its held-out test accuracy. Select on validation first, then run one final test for the chosen model.